# Introduction to Quantum Computing with Qiskit

This notebook starts from zero. You do not need previous experience with quantum physics or linear algebra. You need only basic Python, such as variables, loops, and functions.

We will use a simulator, which is an ordinary program that calculates what a small ideal quantum computer would do. The final topic is a small demonstration of Shor's factoring algorithm for the number 15.

The work time for this notebook is 2-3 hours, so work through as much as you can, take the rest home, and ask whatever questions you may have! 

## What you will learn

| Section | Topic |
|:---:|:---:|
| 1 | Qubits, measurement, probability, and histograms |
| 2 | Gates that change one qubit |
| 3 | Gates that connect qubits |
| 4 | Entanglement |
| 5 | The Quantum Fourier Transform |
| 6 | A small, simulated Shor's algorithm example |

Please only edit code cells that explicitly ask you to make changes, such as exercise cells. Other code cells may be used by later parts of the notebook, so if you accidentally modify or delete one, you can restore it by copying the original cell from the GitHub version of this notebook.



## How to work through this notebook

Run the setup cells, then work through the notebook in order. Pause at exercise cells and complete the marked instructions before running them. Unfinished starters may raise errors or produce incomplete results. If you do not want to work through an exercise yourself, skip its starter cell and run the provided solution cell instead. The optional challenges require you to complete their TODOs before execution.


> 
> **A note before you begin:** This notebook is designed to be comprehensive and accessible even if you are completely new to quantum computing. Starting around **Section 5 (The Quantum Fourier Transform)**, the circuits become significantly more complex, so the explanations also become much more detailed. There is a **lot** of text from this point onward because new concepts are explained as they appear rather than assumed as prior knowledge.
>
> Reading and fully understanding every explanation is **optional** if your main goal is to follow along with the code and see how the algorithms work. However, it is recommended if you want to understand **why** the circuits work rather than simply how to build and run them.
>


## A tiny math primer

A **vector** is an ordered list of numbers. In this notebook, a one-qubit state is represented by two numbers: one for the possibility of measuring 0 and one for the possibility of measuring 1. These numbers are called **amplitudes**. An amplitude can be negative or complex, so it is not itself a probability. To obtain a probability, take its magnitude and square it. This is called the Born rule, a fundamental postulate of quantum mechanics. It is not derived from quantum theory's other basic rules; rather, its predictions are supported by experimental observations.

For example, the state $\frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle$ has amplitude $1/\sqrt{2}$ for each outcome. Each probability is $(1/\sqrt{2})^2 = 1/2$. The probabilities of all possible outcomes must add to 1.

You will see $i$, the imaginary unit, in some gate definitions. It is defined by $i^2=-1$. You do not need to calculate with it by hand here. It lets quantum states carry phase information, which gates can later turn into observable probability changes. Phase is the angle associated with a quantum amplitude; differences in phase determine how amplitudes reinforce or cancel one another when they combine. For example, amplitudes $+1/\sqrt{2}$ and $-1/\sqrt{2}$ each give the same probability, $1/2$, but have opposite phases, so they cancel if added together.

You may also see phase written as $e^{i\theta}$, where $\theta$ is an angle. This is a compact way to describe a rotation in the complex-number plane: $e^{i0}=1$, $e^{i\pi/2}=i$, $e^{i\pi}=-1$, and $e^{i\pi/4}$ is a 45-degree rotation. Multiplying an amplitude by $e^{i\theta}$ changes its angle by $\theta$ without changing its magnitude, so its immediate measurement probability stays the same.

## Reading results: histograms

The bar graphs used throughout this notebook are called **histograms**. Each label along the bottom is a measurement outcome, such as `0`, `1`, or `101`. The height of its bar is the number of times that outcome appeared after repeating the circuit. A circuit is run many times because a single quantum measurement is random whenever more than one outcome has nonzero probability. In Qiskit, one repetition is called a **shot**.

Qiskit prints bit strings with the highest-indexed classical bit on the left. For a two-qubit circuit measured with `measure([0, 1], [0, 1])`, the displayed string `10` means classical bit 1 is 1 and classical bit 0 is 0. The comments in each example point this out when it matters.


## 0. Setup

Run the next cell once in a new notebook environment. It installs Qiskit, the Python toolkit used to describe quantum circuits, and Aer, the simulator used to run them.


In [ ]:
# Install the packages needed by this notebook. Run this once per environment.
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pylatexenc matplotlib==3.11.2

## Install the packages needed by this notebook. Run this once per environment.

In [ ]:
# Imports used throughout the notebook.
from qiskit import QuantumCircuit, transpile
try:
    from qiskit_aer import AerSimulator
    simulator = AerSimulator()
    AER_AVAILABLE = True
except ImportError:
    # This fallback supports the ideal examples if Aer was not installed yet.
    from qiskit.providers.basic_provider import BasicSimulator
    simulator = BasicSimulator()
    AER_AVAILABLE = False
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from math import gcd, pi
from fractions import Fraction

print("All set! Qiskit is ready to go.")
if not AER_AVAILABLE:
    print("Aer was not found, so the ideal examples will use Qiskit's BasicSimulator.")


---
## 1. Qubits, measurement, and your first circuit

### Bits and qubits

A classical **bit** stores exactly one of two values: 0 or 1. A **qubit** also gives 0 or 1 when measured, but before measurement it can be in a **superposition**, a state that assigns amplitudes to both outcomes.

We use **Dirac notation**: $|0\rangle$ means the state that always measures as 0, and $|1\rangle$ means the state that always measures as 1. The labels 0 and 1 inside the kets are just names, chosen to match the classical bit value that state will output when measured, not something derived. A general one-qubit state is

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle.$$

Here $\alpha$ and $\beta$ are amplitudes. Measurement produces 0 with probability $|\alpha|^2$ and 1 with probability $|\beta|^2$. The notation $|z|$ means the magnitude of $z$. The rule $|\alpha|^2+|\beta|^2=1$ makes the total probability equal to 1. 

### Circuits and measurement

A quantum **circuit** is a sequence of instructions applied from left to right. It has quantum wires, called qubits, and classical wires, called classical bits. Measurement transfers one qubit's result into one classical bit and ends the quantum part of that qubit's story for this circuit.


In [ ]:
# Create a circuit with 1 qubit and 1 classical bit
qc = QuantumCircuit(1, 1)

# Measure qubit 0, store the result in classical bit 0
qc.measure(0, 0)

# Draw the circuit
qc.draw("mpl")

**How to read these circuit diagrams:**
- **q** is the qubit line, this is your 1 quantum bit, starting in state |0⟩.
- **c** is the classical bit line, this is where measurement results get stored as ordinary 0s and 1s.
- The box with the **meter/gauge symbol** (⌒/) is the **measurement gate**, it's the operation that reads the qubit's state and converts it into a classical bit.
- The **arrow** pointing from q down to c shows *where* the result goes: qubit 0's measurement result is written into classical bit 0.
- The **"1"** above the c line is the width of the classical register (how many classical bits exist total, here, just 1). The **"0"** below it is the specific bit index being written to, not the value output of the qubit. To see the actual measured value(s), you need to simulate the circuit over many attempts (shots, discussed soon) and plot the results, usually with a histogram (`plot_histogram`), since a single run only gives one outcome and quantum results are inherently probabilistic across many runs.
- The **double diagonal line (⫽)** on the c line is notation meaning "this is a classical wire carrying multiple bits" (even though here it's just 1), it's just how Qiskit distinguishes classical wires from quantum wires visually.

Every qubit begins in $|0⟩$ unless we change it. The next cell repeats a measurement 1,000 times. The histogram should have one bar, `0`, with a height of 1,000.


In [ ]:
# Run the circuit 1000 times on the simulator
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()

print("Results:", counts)
plot_histogram(counts)

**Running the circuit**
- **`transpile(qc, simulator)`** rewrites your circuit into the specific gate set and layout the chosen simulator (or real hardware) actually understands, without changing what the circuit does. Every circuit needs this step before it can run.
- **`simulator.run(compiled, shots=1000)`** executes the compiled circuit 1000 times independently. Each run is called a **shot**. You need many shots because a single measurement only gives you one random outcome, and quantum results are probabilistic, so repeating it many times lets you see the actual probability distribution.
- **`.result()`** collects the outcomes from all 1000 shots into a result object.
- **`.get_counts()`** tallies those outcomes into a dictionary, like `{'0': 507, '1': 493}`, showing how many times each bitstring was measured. This is what you'll typically feed into `plot_histogram()` to visualize.

As expected, every shot returned `0`. The histogram is useful because it shows the distribution of results at a glance. We will next use gates to change that distribution.


---
## 2. Single-qubit gates

A **gate** is an operation on a qubit. Quantum gates are the analog of classical gates that operate on classical bits. Think of it as one instruction in a circuit. Gates are reversible before measurement, meaning that their effect can be undone by another gate.

### X gate, also called NOT

The X gate exchanges the two basis states:

$$X|0\rangle=|1\rangle, \qquad X|1\rangle=|0\rangle.$$

It is the quantum counterpart of a classical NOT operation.


In [ ]:
# Apply an X gate, then measure
qc = QuantumCircuit(1, 1)
qc.x(0)          # Flip qubit 0
qc.measure(0, 0)
qc.draw("mpl")

In [ ]:
# Run the X-gate circuit. Every shot should measure 1.
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()
print("Results:", counts)
plot_histogram(counts)

### Hadamard gate, H

The Hadamard gate changes how the two possible outcomes combine. Applied to a qubit that starts as 0 or 1, it creates an even split between measuring 0 and measuring 1.

Mathematically:

$$H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle+|1\rangle), \qquad H|1\rangle = \frac{1}{\sqrt{2}}(|0\rangle-|1\rangle).$$

These two superposition states appear so often that they have their own names:

$$|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle), \qquad
|-\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle).$$

So we can write the Hadamard operation more simply as

$$H|0\rangle = |+\rangle, \qquad H|1\rangle = |-\rangle.$$

When the H gate acts on a + or - state, it brings it back to a 0 or 1 state, respectively (explained next). Both outputs have the same $1/\sqrt{2}$ size on each term, so $|amplitude|^2 = 1/2$ either way, and an even 50/50 split when measured. The only difference between the two is a sign on the $|1\rangle$ term, plus in the first case, minus in the second.

If a qubit is already in a superposition, H does not simply put it "in superposition again." It combines the parts already there. They can reinforce one outcome and cancel the other, so H can turn an equal superposition into a state that will measure as definitely 0 or definitely 1. For example, applying $H$ to $\frac{1}{\sqrt{2}}(|0\rangle-|1\rangle)$ gives back $|1\rangle$ exactly, the minus sign caused the $|0\rangle$ terms to cancel and the $|1\rangle$ terms to add up.

H itself does not collapse the state; measurement does. Instead, H changes the state so that the later measurement can have a different result.

In [ ]:
# Put a qubit in superposition and measure
qc = QuantumCircuit(1, 1)
qc.h(0)           # Apply Hadamard
qc.measure(0, 0)
qc.draw("mpl")

In [ ]:
# Run the H-gate circuit. The histogram should be close to 50% 0 and 50% 1. It won't be exact!
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()
print("Results:", counts)
plot_histogram(counts)

### Phase gates: Z, S, and T

The Z, S, and T gates change **phase**, which is the angle of an amplitude in the complex-number plane. They leave the $|0\rangle$ amplitude unchanged and rotate the $|1\rangle$ amplitude without changing its measurement probability.

$$Z|1\rangle=-|1\rangle=e^{i\pi}|1\rangle \quad \text{(180-degree rotation)}$$

$$S|1\rangle=i|1\rangle=e^{i\pi/2}|1\rangle \quad \text{(90-degree rotation)}$$

$$T|1\rangle=e^{i\pi/4}|1\rangle \quad \text{(45-degree rotation)}$$

An immediate measurement may show the same histogram, but later gates can use interference to turn these phase differences into different measurement probabilities.

In [ ]:
# H creates equal amplitudes. Z changes the sign of the |1> amplitude.
# The final H lets those amplitudes interfere, so the phase change becomes measurable.
qc_z = QuantumCircuit(1, 1)
qc_z.h(0)
qc_z.z(0)
qc_z.h(0)
qc_z.measure(0, 0)
qc_z.draw("mpl")

In [ ]:
# Simulate the circuit.
compiled = transpile(qc_z, simulator)
result = simulator.run(compiled, shots=1000).result()
counts = result.get_counts()
print("H, then Z, then H:", counts)
plot_histogram(counts)

This circuit applies H, then Z, then H again, and every measurement comes out `1`, with no randomness. Here's why.

The first `h(0)` puts the qubit into an equal mix of $|0\rangle$ and $|1\rangle$: $(|0\rangle+|1\rangle)/\sqrt{2}$. Measuring here would give 0 or 1 with 50/50 probability.

`z(0)` doesn't change those probabilities, it flips the sign on the $|1\rangle$ part only: $(|0\rangle-|1\rangle)/\sqrt{2}$. If you measured right now, you'd still get 50/50, that sign flip is invisible to a direct measurement.

The second `h(0)` is what makes the sign flip visible. Applying H again causes the $|0\rangle$ and $|1\rangle$ parts to combine, and because of that earlier sign flip, they combine into pure $|1\rangle$ instead of back into the original state. This combining effect, where amplitudes add up or cancel out depending on their signs, is called **interference**, and it's why the final measurement is guaranteed to be 1, not random at all.


### Exercise 1: predict, build, and check

For each circuit, predict whether the histogram will show only `0`, only `1`, or a roughly even split. Then run the starter cell. Every circuit must end with `measure(0, 0)` because a simulator cannot put a quantum result into a histogram until it is measured.

1. `X`, then `H`, then measurement, beginning in $|0\rangle$.
2. `H`, then `Z`, then `H`, then measurement, beginning in $|0\rangle$.
3. `H`, then `H`, then measurement, beginning in $|0\rangle$. Hint: $H$ is its own inverse.


In [ ]:
# Exercise 1 starter circuits. These are complete and runnable examples.
# Read the gate order, predict the result, then compare with the printed counts.
def run_one_qubit(circuit, label):
    compiled = transpile(circuit, simulator)
    counts = simulator.run(compiled, shots=1000).result().get_counts()
    print(label, counts)
    return counts

qc1 = QuantumCircuit(1, 1)
qc1.x(0)
qc1.h(0)
qc1.measure(0, 0)
run_one_qubit(qc1, "1. X then H:")

qc2 = QuantumCircuit(1, 1)
qc2.h(0)
qc2.z(0)
qc2.h(0)
qc2.measure(0, 0)
run_one_qubit(qc2, "2. H then Z then H:")

qc3 = QuantumCircuit(1, 1)
qc3.h(0)
qc3.h(0)
qc3.measure(0, 0)
run_one_qubit(qc3, "3. H then H:"); # Semicolon suppresses the duplicate auto-printed return value (housekeeping addition).


### Exercise 1 explanation

Circuit 1 ends in an equal superposition, so it gives a roughly 50/50 split between `0` and `1`. Circuit 2 always returns `1`: the first H creates $|+\rangle$, Z changes it to $|-\rangle$ by flipping the relative phase, and the final H transforms $|-\rangle$ into $|1\rangle$. Circuit 3 always returns `0`, because $HH=I$, where $I$ is the identity operation that changes nothing.

In [ ]:
# Exercise 1 solution visualized as three histograms.
for circuit, label in [(qc1, "X then H"), (qc2, "H then Z then H"), (qc3, "H then H")]:
    counts = run_one_qubit(circuit, label)
    fig, ax = plt.subplots()
    plot_histogram(counts, title=label, ax=ax)
    plt.show()

---
## 3. Gates that connect qubits (multi-qubit gates)

So far, each gate has acted on one qubit at a time. A **multi-qubit gate** acts on two or more qubits together. This lets us do things such as flip one qubit depending on another qubit's value, or exchange the states of two qubits. We will start with a conditional flip, then try the Toffoli and SWAP gates.

### CNOT gate, also called controlled-X or CX

Remember that the X gate flips $|0\rangle$ to $|1\rangle$ and $|1\rangle$ to $|0\rangle$. **CNOT** means **controlled-NOT**: it applies that same flip to one qubit, but only when a second qubit is in $|1\rangle$.

The two qubits have different jobs:

- The **control** determines whether the flip happens. If it is 0, leave the target alone; if it is 1, flip the target.
- The **target** is the qubit that receives the X operation when the control is 1.

For inputs that are definitely 0 or 1, the full rule is:

| Control before | Target before | Control after | Target after |
|:---:|:---:|:---:|:---:|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 |
| 1 | 0 | 1 | 1 |
| 1 | 1 | 1 | 0 |

Notice that the control's value stays the same. The target is **flipped**, not simply set to 1: if both inputs are 1, the target becomes 0. Applying the same CNOT twice undoes the flip and returns the original state.

In Qiskit, `qc.cx(0, 1)` makes qubit 0 the control and qubit 1 the target. The first argument is the control, and the second is the target. In the circuit drawing, a filled dot marks the control and a circled plus marks the target; a line joins them to show that they belong to the same gate.

### Walk through the next two examples

Every qubit starts in $|0\rangle$. In the first circuit below, the control is 0, so CNOT leaves the target at 0. Every shot should give `00`.

In the second circuit, `qc.x(0)` first changes the control to 1. CNOT then flips the target from 0 to 1, so every shot should give `11`.

Both circuits use `qc.measure([0, 1], [0, 1])`, which stores qubit 0's result in classical bit 0 and qubit 1's result in classical bit 1. Remember that Qiskit displays these results as `c1c0`, so the target's result is on the left and the control's result is on the right.

### CNOT and Entanglement Aside:

CNOT applies its flip rule directly to the quantum state, it never checks or collapses anything first. If the control qubit is in superposition, that superposition has multiple terms, and CNOT acts on each term separately, using its normal rule: flip the target if that term's control value is 1, leave it if that term's control value is 0.

For example, if $q_0$ is in superposition $(|0\rangle+|1\rangle)/\sqrt{2}$ and $q_1$ starts at $|0\rangle$, the joint state is $(|00\rangle+|10\rangle)/\sqrt{2}$. CNOT(control=$q_0$, target=$q_1$) leaves the $|00\rangle$ term alone (control is 0) and flips the $|10\rangle$ term to $|11\rangle$ (control is 1), giving $(|00\rangle+|11\rangle)/\sqrt{2}$. (This explanation uses normal `q0q1` ordering, not Qiskit's reverse ordering.)

Now $q_0$ and $q_1$ can't be described separately, measuring one instantly tells you the other. That correlation is **entanglement**, and you'll build it directly in Section 4.
For now, the two examples below use a control that's already definite (0 or 1), so you can check the conditional-flip rule in its simplest form before superposition enters the picture.

In [ ]:
# Two qubits (q0, q1) and two classical result slots (c0, c1).
# Start with q0 = 0, so this CNOT should not change q1.
qc = QuantumCircuit(2, 2)
# cx(control, target): q0 is the control and q1 is the target.
# q1 flips only when q0 is 1; q0 itself is never changed.
qc.cx(0, 1)
# measure([qubits], [classical bits]): each position pairs up, so q0's result goes to c0 and q1's result goes to c1.
# Qiskit prints results as c1c0, so the leftmost displayed bit is q1's result.
qc.measure([0, 1], [0, 1])
print("Control q0 = |0> (so target q1 stays |0>):")
qc.draw("mpl")

In [ ]:
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
plot_histogram(result.get_counts())

As expected, q1 never flips since the control q0 was 0. Both bits report as 0 with certainty, no other outcome is possible here.

In [ ]:
# Now flip the control to |1> first, then CNOT
qc = QuantumCircuit(2, 2)
qc.x(0)            # Set control to |1>
qc.cx(0, 1)        # CNOT: now the target flips!
qc.measure([0, 1], [0, 1])
print("Control = |1>:")
qc.draw("mpl")

In [ ]:
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
plot_histogram(result.get_counts())

Now that `q0` is `1`, CNOT flips `q1` from `0` to `1`, producing `11`. This demonstrates a controlled flip. These qubits are not entangled: both have definite states. In Section 4, we will put the control into superposition and use CNOT to create entanglement.

### Toffoli gate, also called CCX

The Toffoli gate has two controls and one target. It flips the target only when both controls are 1. In Qiskit, `ccx(control_a, control_b, target)` applies it. It is useful because it provides conditional logic with more than one condition.


In [ ]:
# Toffoli: both controls must be |1> to flip the target
qc = QuantumCircuit(3, 3)
qc.x(0)              # Control 1 = |1>
qc.x(1)              # Control 2 = |1>
qc.ccx(0, 1, 2)      # Toffoli: flip qubit 2, so now due to the 2 X gates and the Toffoli, we get |111>, which we see in the histogram output in the next cell.
qc.measure([0, 1, 2], [0, 1, 2])
qc.draw("mpl")

In [ ]:
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
print("Both controls |1> -> target flips to |1>:")
plot_histogram(result.get_counts())

The Toffoli only flips its target when *both* controls are 1, so this is a stricter, two-condition version of the single-control CNOT you saw previously.

### SWAP gate

`swap(a, b)` exchanges the complete states of two qubits. For example, if qubit 0 is $|1\rangle$ and qubit 1 is $|0\rangle$, then

$$|01\rangle \rightarrow |10\rangle$$

using Qiskit's $|q_1q_0\rangle$ ordering. After the SWAP, qubit 0 is $|0\rangle$ and qubit 1 is $|1\rangle$.

It is not a measurement or a copy: it is a reversible exchange. If one qubit is in a superposition, such as $|+\rangle$, and the other is $|0\rangle$, then

$$|0\rangle|+\rangle \rightarrow |+\rangle|0\rangle.$$

The superposition simply moves to the other qubit. Nothing gets measured or collapsed, and in this case the qubits do not become entangled.

In [ ]:
# SWAP: exchange qubit 0 (|1>) and qubit 1 (|0>)
qc = QuantumCircuit(2, 2)
qc.x(0)              # qubit 0 = |1>, qubit 1 = |0>
qc.swap(0, 1)        # Now qubit 0 = |0>, qubit 1 = |1>. What do you think this would output in the histogram as? Remember qiskit's leftmost-first convention!
qc.measure([0, 1], [0, 1])
qc.draw("mpl")

In [ ]:
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1000).result()
plot_histogram(result.get_counts())

Before the swap, q0 held `1` and q1 held `0`. After `swap(0, 1)`, those values trade places, so q0 becomes `0` and q1 becomes `1`. Reading the histogram as c1c0 (Qiskit's leftmost-first convention), `10` means c1=1 and c0=0, exactly the swapped result, with no randomness since no superposition was involved.

### Exercise 2: build a quantum half-adder

A half-adder takes two input bits and outputs their **sum** and **carry**, exactly like adding by hand: 1+1 = 0, carry 1.

Addition without carrying is just XOR. No gate is named "XOR," so your job is to find which gate (or combination) *behaves* like XOR for the sum, and which behaves like the carry.

XOR Review:

| bit A | bit B | A XOR B |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

You have X, CNOT, and Toffoli. Combine them so qubit 2 ends up holding the sum, and qubit 3 ends up holding the carry, given inputs on qubits 0 and 1.

Test all four input combinations below and check against binary addition: 0+0=0 carry 0, 0+1=1 carry 0, 1+0=1 carry 0, 1+1=0 carry 1. This code is given to you.

Once one input pair works, don't copy-paste four circuits. Wrap it in a function that takes `bit0` and `bit1`, using `if` to decide whether to add the `x` gates, then loop over the four pairs. This function has been started for you.

In [ ]:
# Exercise 2: build a quantum half-adder.
# q0, q1 = the two input bits. q2 = sum output. q3 = carry output.
# Don't hardcode four separate circuits, write one function that takes
# bit0 and bit1 as arguments and builds the circuit for those inputs.

def half_adder(bit0, bit1):
    qc = QuantumCircuit(4, 4)

    # Set the two input qubits based on bit0 and bit1.
    if bit0 == 1:
        # ...
    if bit1 == 1:
        # ...

    # TODO: add the gates that make q2 hold the sum and q3 hold the carry.
    # You have qc.cx(control, target) and qc.ccx(control1, control2, target)
    # available. Think about how many gates you need for each output qubit.

    # ...

    qc.measure([0, 1, 2, 3], [0, 1, 2, 3])
    return qc

# Test all four input combinations.
for bit0, bit1 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    circuit = half_adder(bit0, bit1)
    counts = simulator.run(transpile(circuit, simulator), shots=1000).result().get_counts()
    print(f"{bit0} + {bit1}:", counts)

### Exercise 2 explanation

The missing gates are (expand this section after attempting the exercise):

In [ ]:
qc.cx(0, 2)
qc.cx(1, 2)
qc.ccx(0, 1, 3) # Don't run this cell, run the next python cell to see the circuit. 

**Sum (qubit 2):** addition without carrying is XOR, and no gate is directly named XOR, but two CNOTs into the same target reproduce it. Each CNOT flips qubit 2 independently if its own control is 1. If only one input is 1, qubit 2 gets flipped once and lands on 1. If both inputs are 1, qubit 2 gets flipped twice, which cancels back to 0. If neither is 1, qubit 2 is never touched and stays 0. That matches the XOR table.

**Carry (qubit 3):** carry is 1 only when *both* inputs are 1, which is precisely what Toffoli already does by definition, flip the target only if both controls are 1. No combination needed here, just a direct application of the gate.

In [ ]:
# Exercise 2 solution.
def half_adder(bit0, bit1):
    qc = QuantumCircuit(4, 4)

    if bit0 == 1:
        qc.x(0)
    if bit1 == 1:
        qc.x(1)

    qc.cx(0, 2)
    qc.cx(1, 2)
    qc.ccx(0, 1, 3)

    qc.measure([0, 1, 2, 3], [0, 1, 2, 3])
    return qc


for bit0, bit1 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    circuit = half_adder(bit0, bit1)

    # Display the input combination and its circuit.
    print(f"\n{bit0} + {bit1}")
    display(circuit.draw("mpl"))

    # Run the circuit.
    counts = simulator.run(
        transpile(circuit, simulator),
        shots=1000
    ).result().get_counts()

    print("Result:", counts)

### Reading the output

Qiskit displays the measured qubits from highest to lowest index, so the output bitstring is ordered as

$$
q_3q_2q_1q_0
$$

Here, $q_0$ and $q_1$ are the two input bits, $q_2$ stores the **sum**, and $q_3$ stores the **carry**.

Therefore, the four outputs read as:

- **0 + 0:** `0000` → carry = 0, sum = 0, inputs = 00 → result is **00**
- **0 + 1:** `0110` → carry = 0, sum = 1, inputs = 10 → result is **01**
- **1 + 0:** `0101` → carry = 0, sum = 1, inputs = 01 → result is **01**
- **1 + 1:** `1011` → carry = 1, sum = 0, inputs = 11 → result is **10**

---
## 4. Building circuits: entanglement

### The Bell state

**Entanglement** occurs when qubits become connected in a way that cannot be fully described by looking at each qubit separately. Their quantum state must instead be described as a joint state of the qubits. This can produce **correlations**, where measuring one qubit gives us information about the measurement of the other. **Correlation** means that the measurement outcomes are related: knowing the result of one can tell us something about the likely or certain result of the other. Entanglement does not allow information to be sent faster than light.

Why? Each person's measurement result is individually random. You cannot control whether your entangled qubit gives 0 or 1, so you cannot use it to encode a message. The correlation only becomes apparent after the two people compare their results using ordinary classical communication. In short, instantaneous correlation $\neq$ instantaneous communication.

Apply H to qubit 0, then CNOT from qubit 0 to qubit 1, using the qiskit $|q1q0\rangle$ convention:

$$|00\rangle \xrightarrow{H\text{ on }q_0} \frac{|00\rangle + |01\rangle}{\sqrt{2}} \xrightarrow{\mathrm{CNOT}} \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

After H, qubit 0 is in an equal superposition. The CNOT acts on both parts of the superposition: it leaves the $q_0=0$ part unchanged and flips qubit 1 in the $q_0=1$ part, leaving the two possibilities $|00\rangle$ and $|11\rangle$.

Each individual qubit has a 50% chance of being measured as 0 or 1. However, their results are perfectly correlated: if one is measured as 0, the other will also be 0; if one is measured as 1, the other will also be 1. Therefore, we measure either 00 or 11, while 01 and 10 never appear on an ideal simulator.

### Why this is called a Bell state

The state we just created,

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}},$$

is one of the **Bell states**, a set of four two-qubit states that are maximally entangled.

What makes this state interesting is that neither qubit's measurement can be predicted on its own: each has a 50% chance of being measured as 0 or 1. However, the two results are connected. If one qubit is measured as 0, the other will also be measured as 0; if one is measured as 1, the other will also be measured as 1.

So although we cannot predict **which** result we will get, we can predict the **relationship between the two results**: they will always match.

The four Bell states are:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}, \qquad |\Phi^-\rangle = \frac{|00\rangle - |11\rangle}{\sqrt{2}},$$

$$|\Psi^+\rangle = \frac{|01\rangle + |10\rangle}{\sqrt{2}}, \qquad |\Psi^-\rangle = \frac{|01\rangle - |10\rangle}{\sqrt{2}}.$$

In [ ]:
# Create the Bell state Phi-plus
bell = QuantumCircuit(2, 2)
bell.h(0)             # Superposition on qubit 0
bell.cx(0, 1)         # Entangle qubit 1 with qubit 0
bell.measure([0, 1], [0, 1])

bell.draw("mpl")

In [ ]:
# Run it , only '00' and '11' should appear (never '01' or '10')
compiled = transpile(bell, simulator)
result = simulator.run(compiled, shots=1000).result()
plot_histogram(result.get_counts())

Bell states are useful in many important quantum computing and communication protocols. In **quantum teleportation**, a shared Bell pair helps transfer the quantum state of a qubit using entanglement and classical communication. In **superdense coding**, a shared Bell pair allows two classical bits of information to be communicated by sending only one qubit. Bell states are also used in **quantum cryptography**, where their measurement correlations can help establish secure communication, and in **quantum networks**, where Bell pairs can provide entanglement between distant quantum devices (such as quantum computers or processors). They also appear in **quantum error correction**, where entanglement and correlations between qubits are used to detect and correct errors, and in **Bell tests**, which experimentally demonstrate correlations that cannot be explained by local hidden-variable theories (theories where measurement outcomes are determined by pre-existing, locally stored information).

### Exercise 3: related entangled states

1. Build a two-qubit circuit that creates $|\Psi^+\rangle = (|01\rangle + |10\rangle)/\sqrt{2}$. The only measurement results should be `01` and `10`. *(P.S. We recommend you then try creating the two other Bell states.)*
2. Build a three-qubit circuit that creates the GHZ (Greenberger-Horne-Zeilinger) state $(|000\rangle + |111\rangle)/\sqrt{2}$. The only measurement results should be `000` and `111`.

For each state, create the `QuantumCircuit`, add the gates needed to produce the state, and measure all qubits. The simulation code is provided below so you can run your circuits and check whether the measurement results match the expected outcomes.


In [ ]:
# Exercise 3.1: create the |Psi+> Bell state.
qc_bell2 = QuantumCircuit(2, 2)
# Add your gates here.
qc_bell2.measure([0, 1], [0, 1])

# Run Exercise 3.1.
counts_bell = simulator.run(
    transpile(qc_bell2, simulator),
    shots=1000
).result().get_counts()

print("Modified Bell state:", counts_bell)
display(plot_histogram(counts_bell, title="Modified Bell state"))


# Exercise 3.2: create a GHZ state on three qubits.
# Uncomment this section once done with 3.1.
'''
qc_ghz = QuantumCircuit(3, 3)
# Add your gates here.
qc_ghz.measure([0, 1, 2], [0, 1, 2])

# Run Exercise 3.2.
counts_ghz = simulator.run(
    transpile(qc_ghz, simulator),
    shots=1000
).result().get_counts()

print("GHZ state:", counts_ghz)
display(plot_histogram(counts_ghz, title="GHZ state"))
'''

### Exercise 3 explanation

For the first circuit, we begin by creating $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ using H followed by CNOT. Applying X to one of the qubits (doesn't matter which) then flips that qubit in both possibilities: $|00\rangle \rightarrow |01\rangle$ and $|11\rangle \rightarrow |10\rangle$. This gives $|\Psi^+\rangle = (|01\rangle + |10\rangle)/\sqrt{2}$, so measuring the circuit produces only `01` and `10`.

For the GHZ state, H on qubit 0 first creates two possibilities: qubit 0 is 0 or 1. The first CNOT connects qubit 1 to this value, producing the two-qubit pattern $|00\rangle + |11\rangle$. A second CNOT from qubit 0 to qubit 2 extends this pattern to all three qubits, producing $(|000\rangle + |111\rangle)/\sqrt{2}$. Therefore, measuring the circuit produces only `000` and `111`.

The completed circuits and their measurement distributions are shown below.

In [ ]:
# Exercise 3.1 solution: create the |Psi+> Bell state.
qc_bell2_solution = QuantumCircuit(2, 2)
qc_bell2_solution.h(0)
qc_bell2_solution.cx(0, 1)
qc_bell2_solution.x(0)
qc_bell2_solution.measure([0, 1], [0, 1])

# Exercise 3.2 solution: create a three-qubit GHZ state.
qc_ghz_solution = QuantumCircuit(3, 3)
qc_ghz_solution.h(0)
qc_ghz_solution.cx(0, 1)
qc_ghz_solution.cx(0, 2)
qc_ghz_solution.measure([0, 1, 2], [0, 1, 2])


# Show the completed circuits.
display(qc_bell2_solution.draw("mpl"))
display(qc_ghz_solution.draw("mpl"))


# Run each circuit and show its measurement distribution.
for circuit, label in [
    (qc_bell2_solution, "Psi+ Bell state: 01 or 10"),
    (qc_ghz_solution, "GHZ state: 000 or 111")
]:
    counts = simulator.run(
        transpile(circuit, simulator),
        shots=1000
    ).result().get_counts()

    print(label + ":", counts)
    display(plot_histogram(counts, title=label))

---
## 5. The Quantum Fourier Transform, QFT


The **Quantum Fourier Transform (QFT)** is a quantum version of the discrete Fourier transform. Its purpose is to take information stored in one form and rewrite it in a form where **repeating patterns** are easier to detect.

A useful analogy is sound. A musical chord may sound like one complicated signal, but a Fourier transform can describe that same sound in terms of the individual frequencies that make it up. The information has not changed; we have changed the way we represent it.

This matters because a repeating pattern has a **period**: the distance before the pattern repeats. For example,

`1, 0, 0, 1, 0, 0, 1, 0, 0, ...` has a period of 3.

This is why the QFT matters for Shor's algorithm: the central quantum task in Shor's algorithm is finding a period. We will see below how the QFT helps make that period detectable.

### Amplitude and phase

So far, we have mostly focused on the probability of measuring a qubit as 0 or 1. However, a quantum amplitude contains more information than just its size: it also has a **phase**.

We have already seen an example:

$$|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \qquad |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}.$$

If either state is measured directly, there is a 50% chance of getting 0 and a 50% chance of getting 1. However, the states are not identical. The minus sign in $|-\rangle$ represents a difference in **relative phase**.

More generally, we can write a state such as

$$\frac{|0\rangle + e^{i\theta}|1\rangle}{\sqrt{2}}.$$

The factor $e^{i\theta}$ represents a phase rotation by an angle $\theta$. You do not need to calculate complex exponentials by hand here. The important idea is that changing phase does not necessarily change the immediate measurement probabilities, but it changes how amplitudes **interfere** later.

This is something we have already used. Applying H to $|+\rangle$ makes the amplitudes reinforce toward $|0\rangle$, while applying H to $|-\rangle$ makes them reinforce toward $|1\rangle$.

The QFT takes this idea much further: it creates carefully chosen phase relationships so that interference can reveal useful patterns in a quantum state.

### What does the QFT transform?

For $n$ qubits, there are

$$N = 2^n$$

possible computational basis states. A general quantum state can therefore have amplitudes associated with $|0\rangle, |1\rangle, \ldots, |N-1\rangle$.

The QFT transforms those amplitudes into a new set of amplitudes with carefully arranged phases. For a single basis state $|x\rangle$, the transformation is

$$QFT|x\rangle = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{2\pi i xk/N}|k\rangle.$$

You do **not** need to calculate this formula manually for the circuits in this notebook. Its pieces tell us what the QFT is doing:

- $N = 2^n$ is the number of possible basis states.
- $x$ represents the input basis state.
- $k$ runs through every possible output basis state.
- $1/\sqrt{N}$ gives the terms equal magnitude.
- $e^{2\pi i xk/N}$ gives the terms different phases.

So, when applied to a computational basis state, the QFT creates an equal-magnitude superposition with a **specific pattern of phases**.

### We have already seen the smallest QFT

For a single qubit, $N=2$. The QFT transforms the two possible basis states as

$$|0\rangle \rightarrow \frac{|0\rangle+|1\rangle}{\sqrt{2}} = |+\rangle,$$

$$|1\rangle \rightarrow \frac{|0\rangle-|1\rangle}{\sqrt{2}} = |-\rangle.$$

<details>
<summary><b>Optional: see how the QFT formula gives these results</b></summary>

For one qubit, $n=1$, so the number of basis states is $N=2^n=2$. The QFT formula therefore becomes

$$QFT|x\rangle=\frac{1}{\sqrt2}\sum_{k=0}^{1}e^{2\pi i xk/2}|k\rangle.$$

The sum simply tells us to evaluate one term for $k=0$ and another for $k=1$.

For $x=0$:

$$QFT|0\rangle=\frac{1}{\sqrt2}\left(e^{2\pi i(0)(0)/2}|0\rangle+e^{2\pi i(0)(1)/2}|1\rangle\right)$$

$$=\frac{1}{\sqrt2}\left(e^0|0\rangle+e^0|1\rangle\right)$$

$$=\frac{|0\rangle+|1\rangle}{\sqrt2}.$$

For $x=1$:

$$QFT|1\rangle=\frac{1}{\sqrt2}\left(e^{2\pi i(1)(0)/2}|0\rangle+e^{2\pi i(1)(1)/2}|1\rangle\right)$$

$$=\frac{1}{\sqrt2}\left(e^0|0\rangle+e^{i\pi}|1\rangle\right)$$

Using $e^0=1$ and $e^{i\pi}=-1$,

$$=\frac{|0\rangle-|1\rangle}{\sqrt2}.$$

So the general QFT formula really does produce the two transformations shown above.

</details>

But these are exactly the transformations performed by the Hadamard gate:

$$H|0\rangle=|+\rangle, \qquad H|1\rangle=|-\rangle.$$

Therefore, for one qubit,

$$QFT_2 = H.$$

Larger QFT circuits extend this idea across multiple qubits, using Hadamard gates together with controlled phase rotations.

### What goes into a three-qubit QFT?

This section explains in detail how a three-qubit QFT works mathematically. It is not necessary to understand every calculation here to understand or implement the QFT circuit, so feel free to skip ahead to the circuit construction if you are mainly interested in the implementation.

The circuit would have three qubits, so $n=3$ and

$$N=2^3=8.$$

It can therefore take any three-qubit state as input. For a computational basis state, the possible inputs range from $|000\rangle$ to $|111\rangle$.

For example, the simplest input is $|000\rangle$. Since $x=0$, every phase in the QFT formula is $e^0=1$, giving

$$QFT|000\rangle=\frac{1}{\sqrt{8}}\left(|000\rangle+|001\rangle+|010\rangle+|011\rangle+|100\rangle+|101\rangle+|110\rangle+|111\rangle\right).$$

Every basis state has the same amplitude and phase.

For another input, such as $|001\rangle$ ($x=1$ here; $x$ is simply the decimal value represented by the input bitstring, so $|001\rangle$ corresponds to $x=1$, $|010\rangle$ to $x=2$, and so on up to $|111\rangle$, where $x=7$), the output still contains all eight basis states with equal amplitude magnitude, but now their **phases are different**:

$$QFT|001\rangle=\frac{1}{\sqrt{8}}\sum_{k=0}^{7}e^{2\pi i k/8}|k\rangle.$$

Expanding the sum gives

$$QFT|001\rangle=\frac{1}{\sqrt{8}}\left(|000\rangle+e^{i\pi/4}|001\rangle+e^{i\pi/2}|010\rangle+e^{3i\pi/4}|011\rangle-|100\rangle+e^{5i\pi/4}|101\rangle+e^{3i\pi/2}|110\rangle+e^{7i\pi/4}|111\rangle\right).$$

Notice that every term still has amplitude magnitude $1/\sqrt{8}$; what changes from term to term is its **phase**.

The phases do **not** represent how far an output state is from the original input. Instead, they are angles that determine how the amplitudes can interfere later. For $|001\rangle$, the phase increases by $\pi/4$ for each successive basis state:

$$0,\ \frac{\pi}{4},\ \frac{\pi}{2},\ \frac{3\pi}{4},\ \pi,\ \frac{5\pi}{4},\ \frac{3\pi}{2},\ \frac{7\pi}{4}.$$

Phase should be thought of as an angle around a circle, so a larger angle does not mean a state is more important or farther away. For example, $7\pi/4$ is equivalent to $-\pi/4$. The important information is the **pattern of phases across the basis states**. When the QFT acts on states containing periodic structure, these phase patterns cause some amplitudes to reinforce and others to cancel, making information about the period easier to extract.

For example, consider the three-qubit state

$$|\psi\rangle=\frac{1}{2}(|000\rangle+|010\rangle+|100\rangle+|110\rangle).$$

The occupied basis states are $0,2,4,6$, so they are spaced apart by a period of 2.

Consider what happens to the amplitude of the output $|001\rangle$ after the QFT. Since $|001\rangle$ corresponds to $k=1$, the four input states contribute the following phases:

$$x=0:\qquad e^{2\pi i(0)(1)/8}=e^0=1,$$

$$x=2:\qquad e^{2\pi i(2)(1)/8}=e^{i\pi/2}=i,$$

$$x=4:\qquad e^{2\pi i(4)(1)/8}=e^{i\pi}=-1,$$

$$x=6:\qquad e^{2\pi i(6)(1)/8}=e^{3i\pi/2}=-i.$$

When these contributions are added together,

$$1+i-1-i=0.$$

They therefore **cancel through destructive interference**, so the probability of measuring $|001\rangle$ is zero.

For another output, $|100\rangle$, which corresponds to $k=4$, the four input states contribute

$$x=0:\qquad e^{2\pi i(0)(4)/8}=e^0=1,$$

$$x=2:\qquad e^{2\pi i(2)(4)/8}=e^{2\pi i}=1,$$

$$x=4:\qquad e^{2\pi i(4)(4)/8}=e^{4\pi i}=1,$$

$$x=6:\qquad e^{2\pi i(6)(4)/8}=e^{6\pi i}=1.$$

When these contributions are added together,

$$1+1+1+1=4.$$

They therefore **reinforce through constructive interference**, giving that output a large amplitude.

We have only checked two of the eight possible outputs so far. If we perform the same calculation for every value of $k$, we find that the contributions cancel completely for

$$k=1, 2, 3, 5, 6, \text{and } 7$$

while they reinforce for

$$k=0\quad\text{and}\quad k=4.$$

We can therefore calculate the complete QFT of the original periodic state. Remember that the original state contributes a factor of $1/2$, while the QFT contributes a factor of $1/\sqrt{8}$. For $k=0$ and $k=4$, the four phase contributions add to 4, so the amplitude of each surviving output is

$$\frac{1}{2}\frac{1}{\sqrt{8}}(4)=\frac{1}{\sqrt{2}}.$$

All of the other output amplitudes are zero because their phase contributions cancel. Therefore,

$$QFT|\psi\rangle=\frac{1}{\sqrt{2}}\left(|000\rangle+|100\rangle\right).$$

If we now measured the state, the only possible results would be

$$|000\rangle\quad\text{or}\quad|100\rangle,$$

each with probability

$$\left|\frac{1}{\sqrt{2}}\right|^2=\frac{1}{2}.$$

This result is especially important because of where those two peaks occur. The original input had a period of

$$r=2,$$

and there are

$$N=8$$

possible basis states.

To understand what this means, we can think of the amplitudes of the original state as the pattern

$$[1,0,1,0,1,0,1,0].$$

Ignoring normalization, the pattern $[1,0]$ repeats every 2 positions, which is why its period is $r=2$. Since there are 8 positions total, this pattern repeats

$$\frac{N}{r}=\frac{8}{2}=4$$

times across the entire state.

The QFT converts this **period information** into **frequency information**. Instead of representing the fact that the pattern repeats every 2 positions, the QFT produces peaks corresponding to the fact that the pattern repeats 4 times across the 8 positions.

This is exactly what we found: the surviving QFT outputs are $k=0$ and $k=4$. The peak at $k=4$ represents the frequency of 4 repetitions across the input. A peak also appears at $k=0$ because all four input amplitudes contribute positively there, so they reinforce rather than cancel. The important peak for identifying the repeating pattern is $k=4$.
Therefore, the QFT has converted

$$\text{period }2\quad\longrightarrow\quad\text{frequency }4.$$

It is not simply preserving some of the original states and removing others. The outputs $|000\rangle$ and $|100\rangle$ survive because the phase contributions from all four input states interfere constructively at those output indices, while they interfere destructively at the others.

<details>
<summary><b>Optional: see another example with a different period</b></summary>

Consider the three-qubit state

$$|\phi\rangle=\frac{1}{\sqrt{2}}(|000\rangle+|100\rangle).$$

The occupied basis states are $0$ and $4$, so they are spaced apart by a period of 4. Ignoring normalization, its amplitude pattern is

$$[1,0,0,0,1,0,0,0].$$

The pattern $[1,0,0,0]$ repeats every 4 positions. Since there are $N=8$ positions total, it repeats

$$\frac{N}{r}=\frac{8}{4}=2$$

times. We should therefore expect the QFT to reveal a frequency of 2.

Let's calculate the QFT. For each possible output $k$, the two input states $x=0$ and $x=4$ contribute phases

$$e^{2\pi i(0)k/8}=1$$

and

$$e^{2\pi i(4)k/8}=e^{i\pi k}=(-1)^k.$$

If $k$ is even, then $(-1)^k=1$, so the contributions reinforce:

$$1+1=2.$$

If $k$ is odd, then $(-1)^k=-1$, so the contributions cancel:

$$1-1=0.$$

Therefore, the surviving outputs are

$$k=0,\quad2,\quad4,\quad6.$$

Including the normalization factors, the QFT gives

$$QFT|\phi\rangle=\frac{1}{2}\left(|000\rangle+|010\rangle+|100\rangle+|110\rangle\right).$$

Each surviving output therefore has probability

$$\left|\frac{1}{2}\right|^2=\frac{1}{4}.$$

Notice that the surviving peaks are separated by 2:

$$0,\quad2,\quad4,\quad6.$$

This matches

$$\frac{N}{r}=\frac{8}{4}=2.$$

So in this example,

$$\text{input period }4\quad\longrightarrow\quad\text{frequency }2.$$

This is the reverse of our previous example: a pattern that repeats every 2 positions produced peaks separated by 4, while a pattern that repeats every 4 positions produces peaks separated by 2.

</details>

### Why does this phase pattern matter?

For a single input such as $|001\rangle$, the QFT creates a particular phase pattern across the output basis states. The input value determines how quickly that phase changes. From

$$e^{2\pi i xk/N},$$

increasing $x$ makes the phase rotate more quickly as $k$ moves through the output states. In this sense, different input values produce different "phase frequencies."

By itself, this is not yet especially useful. The power of the QFT appears when the input is a **superposition of many basis states**. Each input basis state contributes its own phase pattern to the same output states. Those contributions then add together.

As we saw in the example above, when the input contains a periodic superposition such as

$$|\psi\rangle=\frac{1}{2}(|000\rangle+|010\rangle+|100\rangle+|110\rangle),$$

the occupied basis states are spaced every 2 positions. When the QFT transforms this entire superposition, the phase contributions from these four terms combine at each output state. At some outputs, their amplitudes point in different directions and cancel through **destructive interference**. At others, they line up and add through **constructive interference**.

In this example, that interference transformed

$$\frac{1}{2}(|000\rangle+|010\rangle+|100\rangle+|110\rangle)$$

into

$$\frac{1}{\sqrt{2}}(|000\rangle+|100\rangle).$$

Why is this useful? Suppose we did not already know that the original state had a period of 2. If we measured the original state directly, we would only obtain one of

$$0,\quad2,\quad4,\quad6.$$

A single result does not reveal that these values are spaced every 2 positions because measurement does not let us see the entire superposition at once.

After the QFT, however, the hidden periodic structure affects where the measurement probabilities are concentrated. In this example, the period of 2 produces a frequency peak at $k=4$. Since

$$\frac{N}{r}=4,$$

we can work backwards from this frequency information to obtain information about the original period.

More generally, for a perfectly periodic pattern whose period $r$ divides $N$, the QFT produces peaks separated by $N/r$. The QFT therefore gives us a way to extract information about a period that was previously encoded across many components of a superposition and could not be observed directly with a single measurement.

If $r$ does not divide $N$ exactly, these peaks instead occur near multiples of $N/r$. In Shor's algorithm, these approximate peak locations are measured and classical processing is then used to recover the period $r$.

This is why the QFT matters for Shor's algorithm: Shor's quantum circuit creates a state containing periodic structure, and the QFT converts that hidden structure into measurement outcomes from which we can infer the period.

So the three-qubit QFT does not normally turn one bitstring into another bitstring. It transforms the amplitudes of the input into a new superposition, where interference can make patterns such as periodicity detectable through measurement.

### How do gates create these phase patterns?

We now know what the QFT needs to produce: a superposition with very specific relative phases. The remaining question is how quantum gates can create it.

The gates in the QFT circuit each have a specific job:

**Hadamard gates (H)** create equal superpositions. For example,

$$H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}}.$$

This gives us the equal-magnitude amplitudes that appear in the QFT.

**Controlled-phase gates** then create the required relative phases. A controlled-phase rotation by an angle $\theta$ leaves $|00\rangle$, $|01\rangle$, and $|10\rangle$ unchanged, while

$$|11\rangle\rightarrow e^{i\theta}|11\rangle.$$

It therefore changes phase without flipping either qubit. In Qiskit, we apply one using

`circuit.cp(theta, control, target)`

The QFT uses progressively smaller phase rotations,

$$\frac{\pi}{2},\quad\frac{\pi}{4},\quad\frac{\pi}{8},\ldots$$

so that each qubit receives the particular phase corrections required by the QFT. The circuit repeats the same basic process across the qubits: **create a superposition with H, then add the necessary phase corrections with controlled-phase gates.**

Why do these particular angles appear? They come directly from the phase term in the QFT formula,

$$e^{2\pi i xk/N}.$$

For an $n$-qubit system, $N=2^n$. Consider our three-qubit example, where $N=8$. The phase term is therefore

$$e^{2\pi i xk/8}.$$

The three qubits represent a binary number. For example,

$$|101\rangle \rightarrow 5 = 4+1.$$

The three binary positions have values

$$4,\quad 2,\quad 1.$$

Recall that the QFT phase is

$$e^{2\pi i xk/N}.$$

The important part for us here is the fraction $xk/N$. Because $x$ is built from the values represented by its qubits, each binary place contributes part of $x$ and therefore contributes part of this fraction.

For three qubits, $N=8$. A qubit in the 4's place can contribute 4 to $x$, so its corresponding contribution inside the fraction is $4/8$. Similarly, the 2's place contributes $2/8$, and the 1's place contributes $1/8$.

When these values appear divided by $N=8$ in the QFT phase, they become

$$\frac{4}{8}=\frac{1}{2},\qquad
\frac{2}{8}=\frac{1}{4},\qquad
\frac{1}{8}=\frac{1}{8}.$$

These fractions determine how much each binary position can contribute to the phase. Each position to the right contributes half as much as the previous one.

But these fractions are not themselves angles. The QFT formula places them inside an exponential of the form

$$e^{2\pi i(\text{fraction})}.$$

Earlier, we wrote a phase as $e^{i\theta}$, where $\theta$ is the phase angle. Comparing

$$e^{2\pi i(\text{fraction})}$$

with

$$e^{i\theta},$$

we can see that the corresponding phase angle is

$$\theta=2\pi(\text{fraction}).$$

In other words, a fraction tells us what fraction of one complete $2\pi$ phase rotation is required. A contribution of $1/2$ means half of a full rotation, a contribution of $1/4$ means one quarter of a full rotation, and so on.

The first of these contributions, $1/2$, corresponds to a phase angle of

$$2\pi\left(\frac{1}{2}\right)=\pi.$$

This part is already produced by the Hadamard gate: recall that

$$H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}},$$

while

$$H|1\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}}
=\frac{|0\rangle+e^{i\pi}|1\rangle}{\sqrt{2}}.$$

So the Hadamard already accounts for the $\pi$ phase associated with the first binary contribution.

The controlled-phase gates then add the smaller phase contributions from the other qubits. The next binary contribution is $1/4$, giving

$$2\pi\left(\frac{1}{4}\right)=\frac{\pi}{2}.$$

The next is $1/8$, giving

$$2\pi\left(\frac{1}{8}\right)=\frac{\pi}{4}.$$

With more qubits, the next would be $1/16$, giving

$$2\pi\left(\frac{1}{16}\right)=\frac{\pi}{8},$$

and the pattern continues.

Therefore, the controlled-phase rotations required by the QFT are

$$\frac{\pi}{2},\quad\frac{\pi}{4},\quad\frac{\pi}{8},\ldots$$

Each time the control qubit is one position farther from the target qubit, its required phase correction is half as large. If the control and target are $d$ positions apart, the required angle is

$$\theta=\frac{\pi}{2^d}.$$

In the code below, the distance $d$ is `target - control`, so we calculate the angle using

`angle = pi / (2 ** (target - control))`.

In [ ]:
def build_qft(n_qubits):
    """Build a Quantum Fourier Transform (QFT) circuit on n_qubits."""

    # Create the circuit that will contain the QFT.
    circuit = QuantumCircuit(n_qubits, name="QFT")

    # Build the QFT from the highest-indexed qubit downward.
    # For 3 qubits: target = 2, then 1, then 0.
    for target in reversed(range(n_qubits)):

        # Create an equal superposition on the current target qubit.
        circuit.h(target)

        # Apply the phase corrections from each lower-indexed qubit.
        # For 3 qubits:
        #   target = 2 -> controls 0 and 1
        #   target = 1 -> control 0
        #   target = 0 -> no controls
        for control in range(target):

            # The phase rotation becomes smaller as the distance between
            # the control and target increases:
            #   adjacent qubits -> pi/2
            #   two apart       -> pi/4
            #   three apart     -> pi/8
            angle = pi / (2 ** (target - control))

            # Apply the controlled-phase rotation discussed above.
            # When both selected qubits are |1>, that component of the
            # state receives a phase factor of e^(i*angle).
            circuit.cp(angle, control, target)

    # The QFT construction leaves the qubit order reversed.
    # Swap the outer qubits to restore the conventional output order.
    # For 3 qubits, this swaps qubits 0 and 2.
    for left in range(n_qubits // 2):
        circuit.swap(left, n_qubits - left - 1)

    return circuit


# Build and display the 3-qubit QFT discussed above.
qft3 = build_qft(3)
qft3.draw("mpl")

### Reading the three-qubit QFT circuit

The circuit above is the three-qubit QFT that we just constructed in code. Reading from left to right, we can see the same pattern used repeatedly:

1. An **H gate** creates a superposition on the current target qubit.
2. **Controlled-phase gates** add the relative phases required by the QFT. The labels such as $\pi/2$ and $\pi/4$ show the angle of each phase rotation.
3. The same process is repeated for the remaining qubits.

Notice that qubits farther apart use smaller controlled-phase rotations. As we derived above, each additional position between the control and target halves the required angle:

$$\frac{\pi}{2},\quad\frac{\pi}{4},\quad\frac{\pi}{8},\ldots$$

At the end of the circuit, we see **SWAP gates**. These do not create the Fourier transform itself. The sequence of H and controlled-phase gates naturally produces the QFT output with the qubit positions reversed. For example, in a three-qubit circuit, the position that would normally be read first appears last and vice versa. The SWAP between qubits 0 and 2 reverses these outer positions so that the final result is displayed in the conventional qubit order. The middle qubit does not need to move.

The important idea is therefore not to memorize the circuit. The H gates create superpositions, the controlled-phase gates create the carefully chosen relative phases that produce the required interference, and the final SWAP gates put the resulting qubits in the expected order.

You can also experiment with the circuit yourself. Try changing

`qft3 = build_qft(3)`

to `build_qft(4)`, `build_qft(5)`, or another number of qubits. As the number of qubits increases, notice how the same structure continues: more H gates are required, more controlled-phase gates appear with progressively smaller angles, and the appropriate outer qubits are swapped at the end.

### QFT: main takeaways

Before moving on, here are the main ideas to remember from this section:

- **The QFT is a change of representation.** Like a classical Fourier transform, it can turn information about repeating patterns into information about frequencies.

- **Quantum amplitudes have both magnitude and phase.** Two states can have the same immediate measurement probabilities but behave differently later because their relative phases are different.

- **The QFT uses phase to create interference.** When multiple input amplitudes contribute to the same output, their phases determine whether they reinforce through constructive interference or cancel through destructive interference.

- For $n$ qubits, there are $N=2^n$ computational basis states, and the QFT of a basis state $|x\rangle$ is

  $$QFT|x\rangle=\frac{1}{\sqrt{N}}\sum_{k=0}^{N-1}e^{2\pi i xk/N}|k\rangle.$$

  For a single basis-state input, this produces an equal-magnitude superposition whose relative phases depend on $x$.

- **The one-qubit QFT is simply the Hadamard gate.** Larger QFT circuits extend the same idea using Hadamard gates and controlled-phase rotations.

- **The QFT can reveal periodic structure hidden across a superposition.** A direct measurement only gives one outcome and cannot show us the entire pattern at once. After the QFT, interference concentrates measurement probability at particular output values related to the period.

- In the clean case where a period $r$ divides $N$, the QFT produces peaks separated by

  $$\frac{N}{r}.$$

  If $r$ does not divide $N$ exactly, the peaks instead occur near these values. This is the connection that will eventually allow us to use the QFT for period finding in Shor's algorithm.

- **The controlled-phase angles are not arbitrary.** They come from the phase $e^{2\pi i xk/N}$ in the QFT definition. Because the register represents binary values, the required phase contributions decrease by powers of two, producing rotations such as

  $$\frac{\pi}{2},\quad\frac{\pi}{4},\quad\frac{\pi}{8},\ldots$$

- **The circuit implementation reflects all of these ideas.** Hadamard gates create the required superpositions, controlled-phase gates build the required phase relationships, and SWAP gates correct the final qubit ordering.

The central idea to carry forward is:

> **The QFT uses carefully chosen phases and quantum interference to transform hidden periodic structure into measurement outcomes from which information about the period can be extracted.**

We now understand what the QFT does and how to construct it. Next, we will look at how to reverse this transformation using the **inverse Quantum Fourier Transform**.

### The inverse QFT

Quantum gates are reversible, and the QFT is no exception. The **inverse Quantum Fourier Transform**, written $QFT^\dagger$ or $QFT^{-1}$, undoes the QFT:

$$QFT^{-1}(QFT|\psi\rangle)=|\psi\rangle.$$

To reverse the transformation, the gates are effectively applied in reverse order and the phase rotations are reversed.

The symbol $\dagger$ is called a **dagger**. In linear algebra it represents the conjugate transpose of an operation, but for our purposes you can think of $QFT^\dagger$ as the operation that runs the QFT backward.

Qiskit can automatically construct the inverse of a circuit using `.inverse()`.

In [ ]:
def build_inverse_qft(n_qubits):
    """Build the inverse QFT on n_qubits."""
    return build_qft(n_qubits).inverse()


inv_qft3 = build_inverse_qft(3)
inv_qft3.draw("mpl")

### Checking that the inverse really works

We can test this directly. We will prepare a basis state $|101\rangle$, apply the QFT, then immediately apply the inverse QFT.

If the two circuits truly undo one another, measuring at the end should return the state we started with.

In [ ]:
# Create a 3-qubit circuit with 3 classical bits.
test_qft = QuantumCircuit(3, 3)

# Prepare |101> by flipping qubits 0 and 2 from |0> to |1>.
test_qft.x(0)
test_qft.x(2)

# Add our 3-qubit QFT circuit onto test_qft.
# inplace=True modifies test_qft itself instead of creating a new circuit.
test_qft.compose(build_qft(3), inplace=True)

# Add the inverse QFT immediately after the QFT.
# This should undo the transformation above.
test_qft.compose(build_inverse_qft(3), inplace=True)

# Measure each qubit into its corresponding classical bit.
test_qft.measure([0, 1, 2], [0, 1, 2])

# Run the complete circuit 1000 times and collect the measurement results.
counts = simulator.run(
    transpile(test_qft, simulator),
    shots=1000
).result().get_counts()

print(counts)
plot_histogram(counts)

The circuit began in $|101\rangle$. The QFT changed its representation, and the inverse QFT then reversed that transformation. We therefore measure `101` again.

The intermediate QFT state would generally not look like $|101\rangle$ if we measured it. The point is that the QFT preserves the quantum information while representing it differently, allowing phase and periodic structure to be manipulated before the inverse transformation is applied.

The histogram should contain only `101`, showing that the inverse QFT successfully reversed the QFT and returned the system to its original state.

There is one new Qiskit method in this example: `.compose()`. It allows us to take one quantum circuit and add the operations from another circuit to it. For example,

`test_qft.compose(build_qft(3), inplace=True)`

takes all of the gates in the circuit returned by `build_qft(3)` and adds them to `test_qft`.

The first argument, `build_qft(3)`, is therefore the circuit that we want to add. The parameter `inplace=True` tells Qiskit to modify `test_qft` directly. Without `inplace=True`, `.compose()` would instead return a new circuit containing the combined operations, and we would need to store that returned circuit ourselves.

We then use `.compose()` again to place the inverse QFT after the QFT:

`test_qft.compose(build_inverse_qft(3), inplace=True)`

The resulting circuit therefore performs

$$|101\rangle
\;\longrightarrow\;
QFT|101\rangle
\;\longrightarrow\;
QFT^{-1}(QFT|101\rangle)
\;\longrightarrow\;
|101\rangle.$$

This is why the final measurement returns `101`.

### Connecting this to Shor's algorithm

In the period-finding procedure used in Shor's algorithm, it is typically the **inverse QFT** that appears directly in the quantum circuit. It converts phase information encoded in the quantum state into measurement outcomes from which information about the period can be recovered.

The QFT and inverse QFT contain the same Fourier-transform relationship in opposite directions. Which direction is used depends on how the quantum state has been prepared and where the Fourier transform appears in the algorithm.

Before putting this into Shor's algorithm, let's practice using the QFT and inverse QFT ourselves.

### Exercise 4: QFT round trip

Prepare the three-qubit binary value 5, written `101`. Apply the QFT, then apply the inverse QFT, and finally measure the result. Use .compose() in your code as well as the functions build_qft() and build_inverse_qft().

If the inverse QFT correctly undoes the QFT, what result should you expect to measure?

In [ ]:
# Exercise 4: QFT round trip on |101>.
qc = QuantumCircuit(3, 3)

# Prepare |101>.


# Apply the QFT.


# Apply the inverse QFT.


# Measure all three qubits.


# Run the circuit.
counts = simulator.run(
    transpile(qc, simulator),
    shots=1000
).result().get_counts()

print(counts)
plot_histogram(counts, title="QFT followed by inverse QFT")

### Exercise 4 explanation

To prepare $|101\rangle$, we apply X to qubits 0 and 2, changing them from 0 to 1 while leaving qubit 1 unchanged.

We then apply the QFT, which transforms $|101\rangle$ into a superposition whose information is encoded across its amplitudes and phases. We do **not** measure here, because measurement would collapse this state before we could reverse the transformation.

Next, we apply the inverse QFT. Since the inverse QFT exactly undoes the QFT,

$$QFT^{-1}(QFT|101\rangle)=|101\rangle.$$

We can now measure the three qubits. Because the original state has been recovered, every shot on an ideal simulator should return `101`.

In [ ]:
# Exercise 4 solution: QFT round trip on |101>.
qc_solution = QuantumCircuit(3, 3)

# Prepare |101>.
qc_solution.x(0)
qc_solution.x(2)

# Apply QFT and then inverse QFT.
qc_solution.compose(build_qft(3), inplace=True)
qc_solution.compose(build_inverse_qft(3), inplace=True)

# Measure all qubits.
qc_solution.measure([0, 1, 2], [0, 1, 2])

# Show the completed circuit.
display(qc_solution.draw("mpl"))

# Run and display the result.
counts = simulator.run(
    transpile(qc_solution, simulator),
    shots=1000
).result().get_counts()

print("Expected result: 101. Actual result:", counts)
display(plot_histogram(counts, title="QFT followed by inverse QFT"))

---
## 6. Shor's algorithm: factoring 15

### From the QFT to Shor's algorithm

We now have the key quantum tool we need. We have seen how the QFT can take periodic structure hidden across a quantum state and transform it into measurement outcomes that reveal information about that period.

Shor's algorithm turns that ability into something remarkable: **factoring integers**.

Suppose we are given a number such as

$$N=15$$

We already know that

$$15=3\times5$$

but imagine that we did not know its factors. For sufficiently large numbers, finding those factors can become extremely difficult for known classical algorithms. This matters in practice because the security of widely used public-key cryptosystems such as RSA is connected to the difficulty of factoring very large integers.

Shor's key insight is that factoring can be connected to the period of a particular repeating function.

Suppose we want to factor $N=15$. We first choose a number $a$ such that

$$1<a<N$$

and $a$ shares no factors with $N$. In practice, we can simply choose $a$ and efficiently calculate $\gcd(a,N)$ using the **[Euclidean algorithm](https://en.wikipedia.org/wiki/Euclidean_algorithm)**, even when the numbers are extremely large. We do not need to know the factors of $N$ beforehand.

If

$$\gcd(a,N)=1$$

then $a$ and $N$ share no factors and we can continue. Funny enough, if $\gcd(a,N)>1$, we have accidentally found a factor of $N$ already and do not need the quantum part of Shor's algorithm at all.

For $N=15$, we could choose

$$a=2$$

Since

$$\gcd(2,15)=1$$

we can continue with $a=2$.

We then consider the function

$$f(x)=a^x\bmod N$$

Here, "mod $N$" means we take the remainder after dividing by $N$. For example, $17\bmod5=2$ because dividing 17 by 5 leaves a remainder of 2. For $a=2$ and $N=15$, this becomes

$$f(x)=2^x\bmod15$$

Let's calculate the first few values:

$$
\begin{aligned}
2^0\bmod15 &= 1\\
2^1\bmod15 &= 2\\
2^2\bmod15 &= 4\\
2^3\bmod15 &= 8\\
2^4\bmod15 &= 1\\
2^5\bmod15 &= 2
\end{aligned}
$$

The outputs therefore repeat:

$$1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots$$

The pattern repeats every 4 inputs, so this function has period

$$r=4$$

This is where the connection to factoring appears. If we know this period, then

$$a^r\equiv1\pmod N$$

For our example,

$$2^4\equiv1\pmod{15}$$

which means that $2^4-1$ is divisible by 15. We can therefore write

$$2^4-1=(2^2-1)(2^2+1)=3\times5$$

and the numbers 3 and 5 are exactly the factors we were looking for.

This will not always work quite this directly for every choice of $a$ and $N$, and we will develop the general procedure below. The important connection for now is

$$
\text{factoring }N
\rightarrow
\text{find the period of }a^x\bmod N
\rightarrow
\text{use that period to find factors of }N
$$

So **period finding is not an unrelated problem**. Shor deliberately chooses a periodic function whose period is mathematically connected to the factors of the number we are trying to factor.

The problem is that finding this period can itself become difficult for a classical computer when the numbers become very large. This is where the quantum part of Shor's algorithm enters.

The quantum computer evaluates the periodic function across a superposition of many input values. This creates a quantum state containing the repeating structure of the function. As we learned in the QFT section, simply measuring that state would reveal only one value rather than the overall pattern.

The Fourier-transform machinery allows that repeating structure to affect the probabilities of our measurements. From those measurement results, we can recover information about the period $r$.

The overall strategy is therefore

$$
\text{number to factor}
\rightarrow
\text{periodic function}
\rightarrow
\text{quantum period finding}
\rightarrow
\text{period}
\rightarrow
\text{factors}
$$

We will build this process for $N=15$. Because 15 is deliberately small, we can follow every step and see exactly where the quantum computation enters and how the period ultimately gives us the factors 3 and 5.

### From the period to the factors

Our example above made the final step particularly easy to see because

$$2^4-1=(2^2-1)(2^2+1)=3\times5$$

More generally, once Shor's algorithm has found a suitable even period $r$, we calculate

$$\gcd(a^{r/2}-1,N)$$

and

$$\gcd(a^{r/2}+1,N)$$

These greatest common divisors can reveal non-trivial factors of $N$.

For our example, $a=2$, $r=4$, and $N=15$, so

$$
\gcd(2^{4/2}-1,15)=\gcd(3,15)=3
$$

and

$$
\gcd(2^{4/2}+1,15)=\gcd(5,15)=5
$$

We have therefore recovered

$$15=3\times5$$

The important point is that the quantum computer does **not** directly output the factors 3 and 5. Its difficult job is to help us find the period $r$. Once we know $r$, extracting the factors is a classical calculation.

### Registers and controlled multiplication

We now know exactly what we want the quantum computer to find. For our example,

$$f(x)=2^x\bmod15$$

produces the repeating sequence

$$1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots$$

with period $r=4$. Earlier, we saw why finding this period matters: once we know $r$, we can use classical GCD calculations to recover the factors of 15.

Our goal now is to design a quantum circuit that can extract information about this period.

But there is an immediate problem: the QFT cannot simply look at the equation $2^x\bmod15$ and tell us its period. We first need to **encode the behaviour of this function into a quantum state**. Once the quantum state contains information about the repeating pattern, the Fourier-transform techniques we developed earlier can help us extract it.

To do this, Shor's circuit uses two groups of qubits. A group of qubits used together for a particular purpose is called a **register**.

Our circuit will contain two registers:

* A **three-qubit counting register**. This is the register we will eventually apply the inverse QFT to and measure. Its job is to collect information about the period.

* A **four-qubit work register**. This register is used to calculate and store values of $2^x\bmod15$. Four qubits are enough because they can represent the integers from 0 through 15 in binary. The work register begins in $|0001\rangle=|1\rangle$ rather than $|0000\rangle=|0\rangle$ because we will be performing multiplication. Starting from 0 would always give 0 after multiplication, while starting from 1 allows us to generate the values of the function.

To see why multiplication appears here at all, remember what exponentiation actually means. A power such as

$$2^3$$

is just repeated multiplication:

$$2^3=1\times2\times2\times2$$

If we take the result modulo 15 after each multiplication, starting from 1 gives

$$
1
\rightarrow
2
\rightarrow
4
\rightarrow
8
\rightarrow
1
\rightarrow\cdots
$$

because

$$
\begin{aligned}
(1\times2)\bmod15 &= 2\\
(2\times2)\bmod15 &= 4\\
(4\times2)\bmod15 &= 8\\
(8\times2)\bmod15 &= 1
\end{aligned}
$$

These are exactly the values we calculated earlier for

$$f(x)=2^x\bmod15$$

By repeatedly multiplying by 2 and taking the result modulo 15, the circuit generates the same periodic sequence we studied earlier. **Modular multiplication therefore provides the circuit with a way to encode the values of $f(x)=2^x\bmod15$ into the quantum state**, allowing their periodic structure to be used for period finding.

Remember that our goal is to discover the **period** of the sequence

$$1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots$$

A period describes how the function behaves across different values of $x$. Knowing only one value, such as

$$f(3)=2^3\bmod15=8$$

does not tell us where the sequence repeats. We need information about the function for multiple inputs so that its repeating structure can appear.

This is where superposition becomes useful. We can prepare the counting register in a superposition of different values of $x$, and then use those values to control the modular multiplication. This allows the quantum state to encode the relationship between many inputs $x$ and their corresponding outputs $2^x\bmod15$ at the same time.

This is where the counting register and **controlled operations** enter.

We have already encountered controlled operations with gates such as CNOT: an operation is performed only when its control qubit is $|1\rangle$. Here, instead of conditionally applying a single X gate, the counting qubits conditionally perform modular multiplication on the work register.

There is one additional idea we need. The counting register stores $x$ in binary. With three counting qubits, for example,

$$x=x_0 2^0+x_1 2^1+x_2 2^2$$

where each $x_j$ is either 0 or 1. Therefore,

$$2^x
=
2^{x_0 2^0}
2^{x_1 2^1}
2^{x_2 2^2}$$

This equation explains why the controlled operation depends on whether a counting qubit is $|0\rangle$ or $|1\rangle$. Each counting qubit represents one binary digit of $x$, and that digit determines whether its corresponding power contributes to the exponent.

For example,

$$x=3=011_2$$

means

$$x=0(4)+1(2)+1(1)=2+1$$

so

$$2^3=2^{2+1}=2^2\times2^1$$

The two $1$ bits tell us that the factors $2^2$ and $2^1$ must be included. The $0$ bit tells us that the factor $2^4$ is absent.

This gives us a direct way to build the calculation with controlled operations: **when a counting qubit is $|1\rangle$, apply the multiplication associated with that binary position to the work register. When it is $|0\rangle$, that multiplication is skipped.**

This means that each binary digit of $x$ can control one piece of the multiplication. The three counting qubits can control multiplication by

$$2^{2^0}=2,\qquad 2^{2^1}=4,\qquad 2^{2^2}=16$$

with each multiplication performed modulo 15.

Since

$$16\bmod15=1$$

the corresponding modular multipliers for this example are therefore

$$2,\qquad4,\qquad1$$

Each multiplication is applied only when its corresponding counting qubit is $|1\rangle$. Together, the three controlled operations produce the same result as multiplying by $2^x\bmod15$ for the value of $x$ represented by the counting register.

For example, suppose the counting register represents

$$x=3=011_2$$

Since

$$3=1+2$$

the operations corresponding to the $2^0$ and $2^1$ positions are activated. Starting from 1 in the work register, this gives

$$1\times2\times4\bmod15=8$$

which is exactly

$$2^3\bmod15=8$$

The crucial quantum advantage is that the counting register can be placed into a **superposition of different values of $x$**. The controlled modular multiplications can therefore correlate each possible $x$ with its corresponding value of $2^x\bmod15$ within the same quantum state.

Conceptually, the circuit is creating a state containing relationships of the form

$$
|x\rangle|1\rangle
\rightarrow
|x\rangle|2^x\bmod15\rangle
$$

across the superposition.

This is the connection we need. The repeating sequence we studied classically is now encoded into relationships between the two quantum registers.

For this small $N=15$ example, the modular multiplication operations can be constructed using gates we already know, particularly controlled-SWAP and controlled-X gates. We will build these operations explicitly rather than treating modular multiplication as a mysterious new quantum gate.

The important picture to keep in mind is

$$
\text{superposition of }x
\rightarrow
\text{controlled modular multiplication}
\rightarrow
\text{encode }2^x\bmod15
\rightarrow
\text{periodic structure in the quantum state}
$$
Through these controlled operations, the counting and work registers become correlated in a way that contains information about the periodic behaviour of $2^x\bmod15$.

We can then apply the **inverse QFT to the counting register**. Just as we saw earlier, the Fourier transform allows periodic structure encoded across a quantum state to produce particular patterns in the probabilities of measurement. Those measurements can then give us information about the period $r$.

We do **not** need to measure the work register. Its purpose is to help encode the periodic relationship that ultimately becomes detectable through the counting register.

<details>
<summary><strong>Mathematical summary: encoding 2ˣ mod 15 using the counting and work registers</strong></summary>

<br>

**Objective:**

$$
f(x)=2^x\bmod15
$$

$$
x=x_0 2^0+x_1 2^1+x_2 2^2
$$

$$
2^x
=
2^{x_0 2^0+x_1 2^1+x_2 2^2}
$$

$$
=
2^{x_0 2^0}
2^{x_1 2^1}
2^{x_2 2^2}
$$

$$
=
(2^1)^{x_0}(2^2)^{x_1}(2^4)^{x_2}
$$

$$
2^x\bmod15
=
2^{x_0}4^{x_1}16^{x_2}\bmod15
$$

$$
=
2^{x_0}4^{x_1}1^{x_2}\bmod15
$$

$$
x_j\in\{0,1\}
$$

$$
x_j=0
\Rightarrow
(2^{2^j})^{x_j}=1
$$

$$
x_j=1
\Rightarrow
(2^{2^j})^{x_j}=2^{2^j}
$$

$$
|x_2x_1x_0\rangle|1\rangle
\longrightarrow
|x_2x_1x_0\rangle|2^x\bmod15\rangle
$$

**Example:**

$$
x=3=011_2
$$

$$
x_2=0,\qquad x_1=1,\qquad x_0=1
$$

$$
2^3
=
(2^1)^1(2^2)^1(2^4)^0
$$

$$
=2\cdot4\cdot1
$$

$$
=8
$$

$$
8\bmod15=8
$$

$$
|011\rangle|0001\rangle
\longrightarrow
|011\rangle|1000\rangle
$$

</details>

Before building the complete circuit, let's look more closely at how these controlled modular multiplication operations work.

In [ ]:
def controlled_mult_2_mod15(circuit, control, work_qubits):
    """
    Apply y -> 2y mod 15 to the work register when the control qubit is |1>.
    If the control qubit is |0>, nothing happens to the work register.
    This operation corresponds to the 2^0 place of x:
        2^(2^0) = 2
    so this is the multiplication used when the least-significant
    binary digit of x is 1.
    """
    # Unpack the four work qubits.
    #
    # q0 is the least-significant bit:
    #
    #     q3 q2 q1 q0
    #      8  4  2  1
    #
    # For example:
    #
    #     |0001> = 1
    #     |0010> = 2
    #     |0100> = 4
    #     |1000> = 8
    q0, q1, q2, q3 = work_qubits

    # Multiplication by 2 modulo 15 moves our relevant states as
    #
    #     1 -> 2 -> 4 -> 8 -> 1
    #
    # or, in binary,
    #
    #     0001 -> 0010 -> 0100 -> 1000 -> 0001
    #
    # This is a cyclic shift of the four binary digits.
    #
    # The three controlled-SWAP gates below perform that shift.
    # Because they are controlled-SWAPs, the shift happens only
    # when the control qubit is |1>.
    circuit.cswap(control, q2, q3)
    circuit.cswap(control, q1, q2)
    circuit.cswap(control, q0, q1)


def controlled_mult_4_mod15(circuit, control, work_qubits):
    """
    Apply y -> 4y mod 15 to the work register when the control qubit is |1>.
    If the control qubit is |0>, nothing happens to the work register.
    This operation corresponds to the 2^1 place of x:
        2^(2^1) = 2^2 = 4
    so this is the multiplication used when the second binary
    digit of x is 1.
    """

    # Unpack the same four work qubits.
    q0, q1, q2, q3 = work_qubits

    # Multiplication by 4 modulo 15 moves our relevant states as
    #
    #     1 -> 4
    #     2 -> 8
    #     4 -> 1
    #     8 -> 2
    #
    # or, in binary,
    #
    #     0001 -> 0100
    #     0010 -> 1000
    #     0100 -> 0001
    #     1000 -> 0010
    #
    # This can be implemented by swapping q0 with q2
    # and q1 with q3.
    #
    # Again, the swaps occur only when control = |1>.
    circuit.cswap(control, q0, q2)
    circuit.cswap(control, q1, q3)


# There is no controlled_mult_16_mod15 function because
#
#     2^(2^2) = 2^4 = 16
#
# and
#
#     16 mod 15 = 1.
#
# Multiplication by 1 leaves the work register unchanged,
# so the third counting qubit requires no modular-multiplication gates.


print("Controlled modular multiplication functions are ready.")


# ============================================================
# Visualize the two controlled modular multiplication circuits
# ============================================================

# Each demonstration circuit has five qubits:
#
#     qubit 0     -> control qubit
#     qubits 1-4  -> four-qubit work register
#
# These are demonstrations of the building blocks.
# We will later use these operations inside the full Shor circuit.

control = 0
work_qubits = [1, 2, 3, 4]


# ------------------------------------------------------------
# Controlled multiplication by 2 mod 15
# ------------------------------------------------------------

mult_2_demo = QuantumCircuit(5)

controlled_mult_2_mod15(
    mult_2_demo,
    control,
    work_qubits
)

display(mult_2_demo.draw("mpl"))


# ------------------------------------------------------------
# Controlled multiplication by 4 mod 15
# ------------------------------------------------------------

mult_4_demo = QuantumCircuit(5)

controlled_mult_4_mod15(
    mult_4_demo,
    control,
    work_qubits
)

display(mult_4_demo.draw("mpl"))

Before building the full circuit, let's verify our custom multiplication operations. The work register is initialized to binary `1`. With the control set to `1`, multiplying by 2 should give `0010`, while multiplying by 4 should give `0100`.

In [ ]:
# Verify our controlled multiplication-by-2 operation.
#
# We will start the work register in |0001> = 1 and set the
# control qubit to |1>, which activates the controlled operation.
#
# We therefore expect:
#
#     2 × 1 mod 15 = 2
#
# so the work register should finish in |0010> = 2.


# Create a circuit with:
#   qubit 0    = control qubit
#   qubits 1-4 = four-qubit work register
#   4 classical bits = used to measure the work register
test = QuantumCircuit(5, 4)

# Set the control qubit to |1>.
# This activates the controlled modular multiplication.
test.x(0)

# Initialize the work register to |0001> = 1.
#
# Qubit 1 is q0, the least-significant bit of the work register,
# so flipping it from |0> to |1> stores the number 1.
test.x(1)

# Apply the controlled multiplication:
#
#     |y> -> |2y mod 15>
#
# Since the control is |1> and y = 1, this should produce y = 2.
controlled_mult_2_mod15(
    test,
    control=0,
    work_qubits=[1, 2, 3, 4]
)

# Measure only the four work qubits.
# Their values are stored in the four classical bits.
test.measure(
    [1, 2, 3, 4],
    [0, 1, 2, 3]
)

# Compile the circuit for our simulator.
compiled = transpile(test, simulator)

# Run the circuit 100 times.
result = simulator.run(
    compiled,
    shots=100
).result()

# Every measurement should give 0010, which represents 2.
print("2 * 1 mod 15 =", result.get_counts())

# Expected result:
#
#     {'0010': 100}
#
# because 2 × 1 mod 15 = 2.

In [ ]:
# Verify our controlled multiplication-by-4 operation.
#
# We again start the work register in |0001> = 1 and set the
# control qubit to |1>, activating the controlled operation.
#
# We expect:
#
#     4 × 1 mod 15 = 4
#
# so the work register should finish in |0100> = 4.

# Create a circuit with:
#   qubit 0    = control qubit
#   qubits 1-4 = four-qubit work register
#   4 classical bits = used to measure the work register
test = QuantumCircuit(5, 4)

# Set the control qubit to |1>.
# This activates the controlled modular multiplication.
test.x(0)

# Initialize the work register to |0001> = 1.
# Qubit 1 is the least-significant work bit, so we flip it to |1>.
test.x(1)

# Apply the controlled multiplication:
#
#     |y> -> |4y mod 15>
#
# Since the control is |1> and y = 1, this should produce y = 4.
controlled_mult_4_mod15(
    test,
    control=0,
    work_qubits=[1, 2, 3, 4]
)

# Measure the four work qubits into the four classical bits.
test.measure(
    [1, 2, 3, 4],
    [0, 1, 2, 3]
)

# Compile the circuit for our simulator.
compiled = transpile(test, simulator)

# Run the circuit 100 times.
result = simulator.run(
    compiled,
    shots=100
).result()

# Every measurement should give 0100, which represents 4.
print("4 * 1 mod 15 =", result.get_counts())

# Expected result:
#
#     {'0100': 100}
#
# because 4 × 1 mod 15 = 4.

These two tests act as **sanity checks** for our custom modular multiplication circuits: we give each one the known input $1$ and verify that multiplication by 2 produces $2$ (`0010`) and multiplication by 4 produces $4$ (`0100`). To verify the circuits comprehensively, we could repeat this process for every valid work-register input from $0$ through $14$ and check that each measured result equals the expected value modulo 15.

### Putting the period-finding circuit together

At this point, we have built and tested the individual operations needed to calculate $2^x\bmod15$. We now need to combine them with the counting and work registers to actually search for the period $r$.

Recall our overall goal:

$$
f(x)=2^x\bmod15
$$

$$
1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots
$$

We already know that this sequence has period $r=4$. Our quantum circuit will now try to recover that period from the structure of the function.

The complete period-finding circuit will follow four main steps.

**1. Prepare the work register**

The work register begins in

$$
|0001\rangle=|1\rangle
$$

because our modular multiplications build the values of $2^x\bmod15$ starting from 1.

**2. Put the counting register into superposition**

The three counting qubits begin in

$$
|000\rangle
$$

Applying a Hadamard gate to each gives

$$
\frac{1}{\sqrt8}
\sum_{x=0}^{7}|x\rangle
$$

so the counting register contains all eight possible values

$$
x=0,1,2,\ldots,7.
$$

**3. Apply the controlled modular multiplications**

We now use the operations we just built.

The binary digits of $x$ determine which modular multiplications are applied:

$$
x_0=1
\Rightarrow
\times2\pmod{15}
$$

$$
x_1=1
\Rightarrow
\times4\pmod{15}
$$

$$
x_2=1
\Rightarrow
\times1\pmod{15}
$$

After these operations, the registers are correlated according to

$$
|x\rangle|1\rangle
\longrightarrow
|x\rangle|2^x\bmod15\rangle.
$$

Across the superposition, this encodes the repeating function

$$
1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8.
$$

This is exactly what our previous section was preparing us to do: the custom multiplication circuits are the building blocks that allow the full circuit to encode this periodic function.

**4. Extract the period with the inverse QFT**

At this point the periodic information exists across the quantum state, but measuring the counting register directly would not reveal the period.

We therefore apply the **inverse QFT to the counting register**. Its job is to transform the periodic structure into a pattern of measurement probabilities.

For this example, the resulting measurements concentrate at

$$
000,\qquad010,\qquad100,\qquad110
$$

corresponding to the integers

$$
0,\qquad2,\qquad4,\qquad6.
$$

These peaks contain information about the period. After we build and run the circuit, we will examine exactly how these measured values allow us to recover

$$
r=4
$$

and then use that period to recover the factors of 15.

So the circuit we are about to build has the structure

$$
\boxed{
\text{prepare registers}
\rightarrow
\text{superposition}
\rightarrow
\text{controlled modular multiplication}
\rightarrow
\text{inverse QFT}
\rightarrow
\text{measurement}
}
$$

Let's now assemble these pieces into the complete period-finding circuit.

In [ ]:
def build_inverse_qft(n_qubits):
    """
    Build the inverse QFT for n_qubits.
    build_qft(n_qubits) was defined earlier in the notebook.
    Calling .inverse() reverses that circuit.
    """
    return build_qft(n_qubits).inverse()

def shors_circuit_15():
    """
    Build the quantum period-finding circuit for:

        N = 15
        a = 2
        f(x) = 2^x mod 15

    We use:
        3 counting qubits -> represent x = 0, ..., 7
        4 work qubits     -> represent values 0, ..., 15

    The goal of the quantum circuit is to reveal the period of

        1, 2, 4, 8, 1, 2, 4, 8, ...

    which is r = 4.
    """

    # ---------------------------------------------------------
    # Create the two registers
    # ---------------------------------------------------------

    n_count = 3
    n_work = 4

    # Total:
    #   qubits 0, 1, 2    -> counting register
    #   qubits 3, 4, 5, 6 -> work register
    #
    # We also create 3 classical bits because only the
    # counting register will be measured.
    circuit = QuantumCircuit(
        n_count + n_work,
        n_count
    )

    # Store the indices of the four work qubits.
    work = list(range(n_count, n_count + n_work))


    # ---------------------------------------------------------
    # STEP 1: Prepare the counting register in superposition
    # ---------------------------------------------------------

    # Initially the counting register is |000>.
    #
    # Applying H to each qubit produces an equal superposition
    # of all 3-bit values:
    #
    #     |000>, |001>, |010>, ..., |111>
    #
    # corresponding to:
    #
    #     x = 0, 1, 2, ..., 7
    for qubit in range(n_count):
        circuit.h(qubit)


    # ---------------------------------------------------------
    # STEP 2: Initialize the work register to |0001> = 1
    # ---------------------------------------------------------

    # Every work qubit begins in |0>.
    #
    # work[0] is the least-significant work bit, so applying X
    # changes the work register from:
    #
    #     |0000> -> |0001>
    #
    # We start at 1 because the modular multiplications will
    # build the values of 2^x mod 15 from this initial value.
    circuit.x(work[0])

    # Barrier only separates the preparation stage visually. It makes each section clearly visible on the final circuit.
    # The circuit we are making is quite big. If you've gotten this far, be extremely proud of yourself, genuinely!
    # Anyway, it does not change the quantum state.
    circuit.barrier()


    # ---------------------------------------------------------
    # STEP 3: Encode f(x) = 2^x mod 15
    # ---------------------------------------------------------

    # Recall that the binary expansion of x gives:
    #
    #     x = x0(2^0) + x1(2^1) + x2(2^2)
    #
    # and therefore:
    #
    #     2^x
    #       = (2^(2^0))^x0
    #         (2^(2^1))^x1
    #         (2^(2^2))^x2
    #
    #       = 2^x0 * 4^x1 * 16^x2
    #
    # Modulo 15:
    #
    #     16 mod 15 = 1
    #
    # so the three possible controlled multiplications are:
    #
    #     counting qubit 0 -> multiply by 2 mod 15
    #     counting qubit 1 -> multiply by 4 mod 15
    #     counting qubit 2 -> multiply by 1 mod 15


    # If counting qubit 0 is |1>, multiply the work register
    # by 2 modulo 15.
    #
    # This corresponds to:
    #
    #     2^(2^0) = 2
    controlled_mult_2_mod15(
        circuit,
        control=0,
        work_qubits=work
    )

    # If counting qubit 1 is |1>, multiply the work register
    # by 4 modulo 15.
    #
    # This corresponds to:
    #
    #     2^(2^1) = 4
    controlled_mult_4_mod15(
        circuit,
        control=1,
        work_qubits=work
    )

    # Counting qubit 2 would control multiplication by:
    #
    #     2^(2^2) = 2^4 = 16
    #
    # but:
    #
    #     16 mod 15 = 1
    #
    # so this operation would simply multiply the work register
    # by 1 and leave it unchanged.
    #
    # Therefore, no gates are required for counting qubit 2.


    # At this point, the circuit has created the relationship:
    #
    #     |x>|1> -> |x>|2^x mod 15>
    #
    # across the superposition of x values.
    #
    # The periodic structure of
    #
    #     1, 2, 4, 8, 1, 2, 4, 8
    #
    # is now encoded in the combined quantum state.

    circuit.barrier()

    # ---------------------------------------------------------
    # STEP 4: Apply the inverse QFT to the counting register
    # ---------------------------------------------------------

    # The inverse QFT acts ONLY on counting qubits 0, 1, and 2.
    #
    # It converts the encoded periodic structure into a pattern
    # of measurement probabilities from which we can infer r.
    circuit.compose(
        build_inverse_qft(n_count),
        qubits=range(n_count),
        inplace=True
    )

    circuit.barrier()

    # ---------------------------------------------------------
    # STEP 5: Measure the counting register
    # ---------------------------------------------------------

    # We measure only the counting register.
    #
    # The work register has already served its purpose by helping
    # encode the periodic relationship between x and 2^x mod 15.
    circuit.measure(
        range(n_count),
        range(n_count)
    )

    return circuit


# Build the complete period-finding circuit.
shor_qc = shors_circuit_15()

# Display the circuit so we can see the complete sequence:
#
# preparation
#     -> modular multiplication
#     -> inverse QFT
#     -> measurement
shor_qc.draw("mpl", fold=40)

#### What is actually happening between the two registers?

An important detail in this circuit is that the controlled modular multiplication gates do not directly change the counting qubits $q_0,q_1,q_2$. Instead, they change the **relationship between the counting register and the work register**.

Before modular multiplication, the counting register is in a superposition while the work register always contains $1$:

$$
\frac{1}{\sqrt{8}}\sum_{x=0}^{7}|x\rangle|1\rangle
$$

After the controlled modular multiplications, each value of $x$ becomes correlated with its corresponding value of $2^x\bmod15$:

$$
\frac{1}{\sqrt{8}}\sum_{x=0}^{7}|x\rangle|2^x\bmod15\rangle
$$

For our example, this is

$$
\frac{1}{\sqrt{8}}\big(
|000\rangle|1\rangle+
|001\rangle|2\rangle+
|010\rangle|4\rangle+
|011\rangle|8\rangle+
|100\rangle|1\rangle+
|101\rangle|2\rangle+
|110\rangle|4\rangle+
|111\rangle|8\rangle
\big)
$$

Now the repeating structure becomes visible in the **correlations** between the registers:

$$
|000\rangle,\ |100\rangle
\longleftrightarrow |1\rangle
$$

$$
|001\rangle,\ |101\rangle
\longleftrightarrow |2\rangle
$$

$$
|010\rangle,\ |110\rangle
\longleftrightarrow |4\rangle
$$

$$
|011\rangle,\ |111\rangle
\longleftrightarrow |8\rangle
$$

The counting-register values associated with the same work-register value are separated by

$$
4
$$

which is exactly the period $r$.

This is the important connection between the work register and the measurement we eventually make. The work-register values group together counting states that are separated by the period. For example, $|000\rangle$ and $|100\rangle$ are both correlated with the work value $|1\rangle$, while $|001\rangle$ and $|101\rangle$ are both correlated with $|2\rangle$. Every group has the same spacing of 4.

The inverse QFT acts on the counting register while these correlations still exist. During the inverse QFT, the different counting-state amplitudes interfere with one another. For this particular periodic structure, the interference causes some possible counting-register outcomes to cancel while others reinforce each other.

After the inverse QFT, the only counting-register outcomes with non-zero probability are

$$
|000\rangle,\qquad
|010\rangle,\qquad
|100\rangle,\qquad
|110\rangle
$$

and each occurs with probability

$$
\frac14.
$$

So the sequence of events is

$$
\text{period }4
\rightarrow
\text{counting states grouped with spacing }4
\rightarrow
\text{inverse-QFT interference}
\rightarrow
000,\ 010,\ 100,\ 110
$$

This explains why we do not need to measure the four work qubits. Their role is to establish the correlations that give the counting register its periodic structure. Once those correlations have been created, the work register can remain unmeasured.

The inverse QFT transforms that periodic structure into a pattern that we can observe by measuring only the counting register.

So the key idea is

$$
\boxed{
\text{modular multiplication}
\rightarrow
\text{correlations between the registers}
\rightarrow
\text{periodic structure}
\rightarrow
QFT^{-1}
\rightarrow
\text{measurement peaks}
}
$$

The work register therefore does not need to send a value back into the counting register or be measured separately. Its interaction with the counting register has already created the structure that determines how the counting-register amplitudes interfere during the inverse QFT.

Our next task is to run the circuit, observe these measurement peaks, and understand how the measured values allow us to recover the period $r=4$.

### Run the period-finding circuit

Now let's simulate the complete circuit and check whether the inverse QFT produces the measurement pattern we predicted from the period-$4$ structure.

When we run the circuit, we measure **only the three counting qubits** $q_0,q_1,q_2$. The four work qubits are left unmeasured because their role was to create the correlations that encoded the periodic structure before the inverse QFT. Therefore, every result in our histogram will be a three-bit string such as `000`, `010`, or `110`.

Why are we using three counting qubits? Three qubits provide

$$
2^3=8
$$

possible counting states, representing $x=0$ through $7$. For this small example, that range contains two complete repetitions of our period-$4$ function:

$$
1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8
$$

which allows the inverse QFT to reveal the period cleanly.

For larger factoring problems, we generally use more counting qubits to obtain enough precision to recover the period. A common choice in Shor's algorithm is roughly **twice as many counting qubits as there are bits in $N$**. For example, if we wanted to factor

$$
N=21
$$

then

$$
21=10101_2
$$

requires 5 bits to represent, so a typical full Shor implementation might use about

$$
2(5)=10
$$

counting qubits.

Our three-qubit choice for $N=15$ is therefore a deliberately small, clean demonstration chosen because its period fits especially neatly into the counting register.

The exact bar heights vary because measurement is sampled. The important feature is the **locations of the peaks**. A clean simulation should concentrate nearly all results in `000`, `010`, `100`, and `110`, with each occurring approximately $25\%$ of the time.

After observing these peaks, we will see how the measured values can be used to recover the period $r=4$.

In [ ]:
# Run the circuit!
compiled = transpile(shor_qc, simulator)
result = simulator.run(compiled, shots=2048).result()
counts = result.get_counts()

print("Measurement results:")
print(counts)
plot_histogram(counts)

### Interpret the measurements

We have now reached the output of the quantum part of Shor's algorithm. The counting register has three qubits, so each measured bit string represents an integer $s$ from $0$ to $7$. Since

$$
2^3=8,
$$

we interpret a measurement as the fraction

$$
\frac{s}{8}.
$$

The peaks we observed were

$$
000,\qquad010,\qquad100,\qquad110,
$$

corresponding to

$$
s=0,\qquad2,\qquad4,\qquad6.
$$

Consider the measurement `010`. It represents

$$
\frac{s}{8}
=
\frac{2}{8}
=
\frac14.
$$

The denominator $4$ gives us the candidate period

$$
r=4.
$$

Similarly, measuring `110` gives

$$
\frac68=\frac34,
$$

which also has denominator $4$.

In a general run of Shor's algorithm, the measured fraction may only be **close** to a fraction whose denominator is the period. A classical method called **continued fractions** is used to find a likely small-denominator fraction from the measurement. Our $N=15$ example is especially clean, so the useful measurements reduce directly to fractions with denominator $4$.

Not every measurement gives the complete period immediately. `000` gives $0/8=0$, which provides no useful candidate, while `100` gives

$$
\frac48=\frac12,
$$

giving the candidate $r=2$. We can test a candidate period classically by checking whether

$$
2^r\bmod15=1.
$$

For $r=2$,

$$
2^2\bmod15=4\neq1,
$$

so $2$ is rejected. For $r=4$,

$$
2^4\bmod15=1,
$$

so we have successfully recovered the period

$$
\boxed{r=4}.
$$

At this point, the **quantum period-finding part is finished**. The remaining calculation is classical.

Because $r$ is even, we calculate

$$
\gcd\left(2^{r/2}-1,15\right)
\qquad\text{and}\qquad
\gcd\left(2^{r/2}+1,15\right).
$$

Substituting $r=4$ gives

$$
\gcd(2^2-1,15)=\gcd(3,15)=3
$$

and

$$
\gcd(2^2+1,15)=\gcd(5,15)=5.
$$

Therefore,

$$
\boxed{15=3\times5}.
$$

The next code cell performs this final measurement interpretation and classical factor-recovery step automatically.

In [ ]:
def interpret_results(counts, n_count, N, a):
    """Interpret the quantum measurements and recover factors of N."""

    denominator = 2 ** n_count
    factors_found = set()

    print(f"Factoring N={N} with a={a}")
    print(f"Counting-register denominator: {denominator}\n")

    # Convert each measured peak into a candidate period.
    for output_bits, count in sorted(counts.items(), key=lambda item: -item[1]):
        s = int(output_bits, 2)

        if s == 0:
            print(f"{output_bits}: {count} shots, no period information")
            continue

        fraction = Fraction(s, denominator).limit_denominator(N)
        r = fraction.denominator

        print(
            f"{output_bits}: {count} shots, "
            f"{s}/{denominator} = {fraction}, candidate r={r}"
        )

        # Check whether the candidate period can be used to extract factors.
        if r % 2 != 0:
            print("  Candidate period is odd, so skip it.")
            continue

        if pow(a, r, N) != 1:
            print("  Candidate does not satisfy the period condition, so skip it.")
            continue

        factor_1 = gcd(a ** (r // 2) - 1, N)
        factor_2 = gcd(a ** (r // 2) + 1, N)

        for factor in (factor_1, factor_2):
            if 1 < factor < N:
                factors_found.add(factor)

    # Display the final result.
    factors = sorted(factors_found)

    if len(factors) >= 2:
        print(f"\nFactors found: {factors[0]} x {factors[1]} = {N}")
    else:
        print("\nNo complete factor pair was recovered from these samples.")

    return factors


factors = interpret_results(counts, n_count=3, N=15, a=2)

### Why a useful measurement can give only part of the period

A QFT measurement gives an estimate of a fraction $s/r$. Continued fractions recovers this fraction in reduced form. If $s$ and $r$ share a factor, the reduced denominator may therefore be a **divisor of the true period**, rather than the full period. For example, $2/4 = 1/2$, so a measurement corresponding to $s=2$ and $r=4$ may produce the candidate denominator 2 even though the true period is 4.

Our helper skips candidate periods $r$ that fail the condition $a^r \bmod N = 1$. This keeps the example simple, but some skipped measurements still contain useful information.

**Optional extension:** Before rejecting a denominator $d$, test its multiples $d, 2d, 3d, \ldots$, stopping at $N$. Use the first candidate $r$ satisfying $a^r \bmod N = 1$, then apply the existing even-period and factor checks. For $a=2,\ N=15$, denominator 2 leads to candidate $r=4$; for $a=2,\ N=21$, denominator 3 leads to candidate $r=6$.

This bounded search is useful for these small teaching examples. Testing many multiples is not an efficient general solution for large integers.

### What just happened?

The quantum computer did not directly output the factors $3$ and $5$. Its job was to solve the part of the factoring problem that Shor's algorithm can accelerate: **finding the period of a modular function**.

For our example, we wanted to factor

$$
N=15
$$

and chose

$$
a=2.
$$

This gave us the periodic function

$$
f(x)=2^x\bmod15,
$$

whose values repeat as

$$
1,\ 2,\ 4,\ 8,\ 1,\ 2,\ 4,\ 8,\ldots
$$

with period

$$
r=4.
$$

The quantum circuit encoded this repeating structure into correlations between the counting and work registers. The inverse QFT transformed that periodic structure into measurable peaks in the counting register. From useful measurements such as `010` and `110`, we obtained

$$
\frac{2}{8}=\frac14
\qquad\text{or}\qquad
\frac{6}{8}=\frac34,
$$

which allowed us to recover the candidate period

$$
r=4.
$$

In a general application of Shor's algorithm, the measured fraction will often only approximate a fraction related to the period, so a classical continued-fraction calculation is used to recover a candidate $r$. The candidate is then checked by verifying

$$
a^r\bmod N=1.
$$

Once we had a valid even period, the quantum computer's job was complete.

Because

$$
a^r\equiv1\pmod N,
$$

we have

$$
a^r-1\equiv0\pmod N.
$$

For an even $r$,

$$
a^r-1
=
(a^{r/2}-1)(a^{r/2}+1).
$$

This allows classical greatest-common-divisor calculations to reveal factors of $N$. In our example,

$$
\gcd(2^2-1,15)=3
$$

and

$$
\gcd(2^2+1,15)=5,
$$

giving

$$
\boxed{15=3\times5}.
$$

So Shor's algorithm is a hybrid quantum-classical algorithm. The quantum computation finds information about the period, while classical computation interprets the measurement, verifies the candidate period, and extracts the factors.

---

### The whole algorithm from beginning to end

Stepping back, the complete process we built was

$$
\boxed{
N
\rightarrow
\text{choose }a
\rightarrow
f(x)=a^x\bmod N
\rightarrow
\text{quantum period finding}
\rightarrow
r
\rightarrow
\text{classical GCDs}
\rightarrow
\text{factors of }N
}
$$

Inside the quantum period-finding step, we:

1. created a **counting register** and a **work register**,
2. placed the counting register into superposition,
3. used controlled modular multiplication to encode $a^x\bmod N$ into the work register,
4. created correlations containing the periodic structure of the function,
5. applied the inverse QFT to the counting register,
6. measured the counting register and observed peaks related to the period,
7. converted those measurements into a candidate period.

Along the way, we used many of the central ideas of quantum computing: qubits, superposition, multi-qubit states, controlled operations, entanglement and correlations, interference, measurement, the quantum Fourier transform, and hybrid quantum-classical computation.

The number $15$ was chosen because all of these ideas can be made visible using a tiny circuit. Shor's algorithm becomes important when $N$ is enormously larger.

---

### What would change with an ideal quantum computer?

Imagine that we had a perfect, large-scale quantum computer whose qubits never decohered (lost the delicate quantum state needed for computation), whose gates never made mistakes, and whose quantum states could remain intact for as long as necessary.

We could then replace our tiny

$$
N=15
$$

with a number hundreds or thousands of bits long, including the kinds of integers used by RSA.

Conceptually, the algorithm would still look remarkably familiar. We would choose an appropriate value of $a$, create large counting and work registers, compute

$$
a^x\bmod N
$$

quantum mechanically, use quantum period finding to recover $r$, and perform the final GCD calculations classically.

The enormous difference would be **scale**.

Our modular multiplication operations were small enough that we could explicitly construct them using a few controlled SWAP gates. For an RSA-sized integer, modular exponentiation would require very large reversible arithmetic circuits containing enormous numbers of quantum gates.

Our three-qubit counting register would become a much larger register capable of resolving the period with enough precision. Our four-qubit work register would become large enough to hold RSA-sized values. The inverse QFT would likewise operate over a much larger register.

We also deliberately chose an example where the period fits perfectly into our eight possible counting states. A real execution would generally produce measurements that approximate fractions related to the period, making the continued-fraction step genuinely necessary.

Still, with perfect physical qubits, these differences would primarily be engineering differences of **size and circuit complexity**. The mathematical structure you implemented in this notebook would remain recognizable.

That is the extraordinary theoretical result behind Shor's algorithm: factoring numbers that become extraordinarily difficult for known classical algorithms can be transformed into a period-finding problem that a sufficiently powerful quantum computer can solve efficiently.

---

### What would change on a realistic quantum computer?

Unfortunately, real qubits are not perfect.

Physical qubits interact with their environment, lose quantum information, and experience imperfect operations. A computation large enough to attack RSA would require an enormous number of quantum operations while preserving delicate quantum information throughout the calculation.

This is why **quantum error correction** becomes essential.

Rather than trusting one physical qubit to store one important piece of quantum information, many physical qubits can be combined to encode a much more reliable **logical qubit**. Errors can then be detected and corrected while the computation continues.

That changes the hardware picture dramatically.

A realistic RSA-scale implementation would require physical qubits for the computation itself, additional physical qubits for quantum error correction, repeated error-detection operations, fault-tolerant logical gates, additional resources for difficult operations, and a classical control system capable of decoding errors and coordinating the quantum processor in real time.

The exact resource requirements are still an active research problem, and improvements in algorithms, error-correcting codes, hardware, and computer architecture continue to change the estimates.

For perspective, a 2025 resource estimate found that RSA-2048 could theoretically be factored in less than a week using fewer than one million noisy physical qubits under its assumed hardware parameters. More recent proposed architectures have produced even lower theoretical estimates. These are resource estimates for machines that would have to satisfy demanding assumptions about error rates, operation speeds, connectivity, error correction, and control. They are not demonstrations that RSA-2048 has been factored by a quantum computer.

This gap between a circuit that works mathematically and a physical machine capable of executing it reliably is one of the central challenges of modern quantum computing.

There is good reason for optimism, however. Quantum error correction has progressed from a theoretical requirement into something researchers can experimentally build and test. Better error-correcting codes, improved physical qubits, more efficient implementations of Shor's algorithm, improved modular arithmetic, and new quantum-computer architectures can all reduce the resources required.

The challenge is enormous, but the target is not mysterious. We know the algorithm. We know why it works. We know many of the requirements for executing it. The open challenge is building a sufficiently large, sufficiently accurate, fault-tolerant quantum computer capable of carrying the computation through to completion.

---

### Q-Day

This leads to a term you may encounter frequently in quantum computing and cybersecurity: **Q-Day**.

Q-Day refers broadly to the point at which a cryptographically relevant quantum computer becomes capable of breaking important currently deployed public-key cryptography using quantum algorithms such as Shor's algorithm.

RSA is particularly relevant here because its security depends on the practical difficulty of factoring large integers. A sufficiently capable fault-tolerant quantum computer running Shor's algorithm would undermine that assumption.

Q-Day does **not** mean that every form of encryption suddenly becomes useless. Different cryptographic systems are affected differently by quantum algorithms. The immediate concern is especially important for quantum-vulnerable public-key systems such as RSA and elliptic-curve cryptography.

This is also why the world is not simply waiting for Q-Day to arrive. **Post-quantum cryptography (PQC)** uses cryptographic problems designed to remain secure against known attacks from both classical and quantum computers. Organizations and standards bodies are already working on the transition to these systems.

If you want to follow estimates and developments surrounding this milestone, **[Is it Q-Day?](https://isitqday.com/)** tracks developments in quantum computing and their possible implications for cryptography.

Any predicted date for Q-Day should be treated as an estimate rather than a known deadline. Nobody currently knows exactly when, or whether on a particular predicted schedule, hardware capable of cryptographically relevant quantum computation will be achieved.

---

### Congratulations!

You made it.

At the beginning of this notebook, concepts such as superposition, controlled gates, quantum interference, the QFT, modular exponentiation, period finding, and Shor's algorithm may have looked like separate and fairly abstract pieces of quantum computing.

You have now connected those pieces into an actual algorithm.

You built quantum states and manipulated their amplitudes. You learned how the QFT can expose periodic structure. You constructed controlled modular multiplication circuits. You used two quantum registers whose correlations encoded a mathematical function. You applied the inverse QFT, measured its output, interpreted the resulting peaks, recovered a period, and finally converted that period into the prime factors of an integer.

Factoring $15$ is a small demonstration. The ideas you used to do it are the same ideas that make Shor's algorithm one of the most important results in quantum computing.

If you made it through the notebook, ran the circuits, understood what the registers were doing, and followed how those three measured qubits ultimately led us from $15$ to $3\times5$, then you built Shor's algorithm from the ground up and understood the ideas that make it work. You should be proud.

The challenges below are completely optional. If you want to keep going, they give you a chance to modify, extend, and experiment with what you have built.

---
## 7. Further experiments

Everything below is optional. These challenges build on the period-finding algorithm you just completed and give you a chance to modify it, experiment with its behaviour, explore noise and larger circuits, and eventually extend Shor's algorithm beyond the original $N=15$ example.

---
### Challenge A: try a different base

Our implementation factored $15$ using

$$
a=2.
$$

Shor's algorithm does not require us to choose $a=2$. We can choose other values satisfying

$$
1<a<N
$$

and

$$
\gcd(a,N)=1.
$$

For this challenge, use

$$
\boxed{a=7}.
$$

Start classically. Calculate

$$
7^x\bmod15
$$

for several consecutive values of $x$ and determine its period.

Fill in the table:

| $x$ | $7^x\bmod15$ |
|:---:|:---:|
| 0 | |
| 1 | |
| 2 | |
| 3 | |
| 4 | |
| 5 | |

What period $r$ do you observe?

Once you know the period, remember how modular exponentiation was controlled by the binary digits of $x$:

$$
7^x
=
(7^{2^0})^{x_0}
(7^{2^1})^{x_1}
(7^{2^2})^{x_2}.
$$

Calculate the three values

$$
7^{2^0}\bmod15,\qquad
7^{2^1}\bmod15,\qquad
7^{2^2}\bmod15.
$$

These tell you which modular-multiplication operations the three counting qubits need to control.

In [ ]:
# Challenge A.1
# Explore the function 7^x mod 15.

N = 15
a = 7

for x in range(8):
    # TODO: calculate a^x mod N
    value = ...
    print(f"x={x}: {value}")

Now build the two modular-multiplication operations required for $a=7$.

You do **not** need to design arbitrary modular arithmetic. We only need the transformations that occur in this particular $N=15$ example.

Use the cells below to experiment with SWAP and X gates until the work register undergoes the required permutations.

You may find it useful to write the relevant decimal values in 4-bit binary first and look for a pattern in how the bits move.

In [ ]:
def controlled_mult_7_mod15(circuit, control, work_qubits):
    """
    Apply |y> -> |7y mod 15> when the control qubit is |1>.
    """

    q0, q1, q2, q3 = work_qubits

    # TODO:
    # Build the controlled multiplication-by-7 operation.
    pass


def controlled_mult_4_mod15_a7(circuit, control, work_qubits):
    """
    Apply |y> -> |4y mod 15> when the control qubit is |1>.
    """

    q0, q1, q2, q3 = work_qubits

    # TODO:
    # Build the controlled multiplication-by-4 operation.
    pass

Finally, modify the period-finding circuit.

The register sizes, initialization, inverse QFT, and measurement can remain the same as in the $a=2$ circuit. The part that must change is the controlled modular exponentiation.

After completing the circuit, run it and inspect its histogram.

**Questions to answer:**

- What measurement peaks do you observe?
- What period do they indicate?
- Do you still recover the factors $3$ and $5$?

In [ ]:
def shors_circuit_15_a7():
    n_count = 3
    n_work = 4

    circuit = QuantumCircuit(n_count + n_work, n_count)
    work = list(range(n_count, n_count + n_work))

    # Prepare the counting register.
    for qubit in range(n_count):
        circuit.h(qubit)

    # Prepare the work register in |1>.
    circuit.x(work[0])

    # TODO:
    # Apply the controlled modular multiplications required for a=7.


    # Apply the inverse QFT.
    circuit.compose(
        build_inverse_qft(n_count),
        qubits=range(n_count),
        inplace=True
    )

    # Measure only the counting register.
    circuit.measure(range(n_count), range(n_count))

    return circuit


# TODO: build, simulate, and plot your circuit.

shor_a7 = ...
counts_a7 = ...

print(counts_a7)
# plot_histogram(...)

---

### Challenge B: use more counting qubits

Our original circuit used three counting qubits, giving

$$
2^3=8
$$

possible measurement values.

What happens if we use more?

With $n$ counting qubits, a measurement $s$ represents the fraction

$$
\frac{s}{2^n}.
$$

Increasing $n$ gives the quantum period-finding procedure more possible fractions and therefore greater precision. This becomes especially important when the period does not fit neatly into a power-of-two-sized counting register.

Our period-$4$ example is unusually clean because $4$ divides $2^n$ for every $n\ge2$. That makes it a useful place to observe how the **same period appears at different register sizes**.

Complete the function below so that it works for any reasonable number of counting qubits.

For $a=2$,

$$
2^{2^j}\bmod15
$$

tells you which modular multiplication is controlled by counting qubit $j$.

You already have the multiplication-by-2 and multiplication-by-4 functions from the main notebook.

If the required multiplier is $1$, no operation is necessary.

In [ ]:
def shors_circuit_15_flexible(n_count):
    """
    Build the a=2, N=15 period-finding circuit
    using n_count counting qubits.
    """

    n_work = 4
    circuit = QuantumCircuit(n_count + n_work, n_count)
    work = list(range(n_count, n_count + n_work))

    # Prepare counting register.
    for qubit in range(n_count):
        circuit.h(qubit)

    # Prepare work register in |1>.
    circuit.x(work[0])

    # Build controlled modular exponentiation.
    for qubit in range(n_count):

        power = pow(2, 2 ** qubit, 15)

        # TODO:
        # power tells you which multiplication operation
        # this counting qubit should control.
        #
        # Handle power == 2
        # Handle power == 4
        # Handle power == 1


    # TODO: apply the inverse QFT.


    # TODO: measure the counting register.


    return circuit

Run your function using

$$
n=3,\qquad n=4,\qquad n=5.
$$

Compare the histograms.

Before running the cell, predict where the period-$4$ peaks should appear.

For example, with four counting qubits there are

$$
2^4=16
$$

possible measured integers. If the peaks represent the fractions

$$
0,\quad\frac14,\quad\frac24,\quad\frac34,
$$

which four integers should appear?

In [ ]:
for n_count in [3, 4, 5]:

    # TODO: build the circuit.
    circuit = ...

    # TODO: simulate it.
    counts = ...

    print(f"{n_count} counting qubits:")
    print(counts)

    # TODO: display its histogram.

**Think about your results:**

Did increasing the number of counting qubits change the period?

Did the number of major peaks change?

How did the integer locations of those peaks change as $2^{n}$ became larger?

Why would the extra precision become much more important for a period that does **not** divide $2^n$ exactly?

---

### Challenge C: what happens when the gates are noisy?

Everything so far has used an ideal simulator. Every gate behaved exactly as requested.

Physical quantum computers experience errors. One simple way to explore their effect is **depolarizing noise**, where a gate has some probability of introducing a random quantum error.

The helper function below builds a noisy simulator for you. You do not need to understand the implementation of the noise model for this challenge.

Your task is experimental:

1. run the same Shor circuit at several error rates,
2. compare the resulting histograms,
3. measure how much probability remains concentrated at the expected period-$4 peaks.

For our three-qubit $a=2$ circuit, the ideal peak locations are

$$
000,\qquad010,\qquad100,\qquad110.
$$

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

def run_with_noise(circuit, error_rate=0.01, shots=2048):
    """
    Run a circuit using a simple depolarizing gate-noise model.
    """

    noise_model = NoiseModel()

    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(error_rate, 1),
        ["h", "x"]
    )

    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(min(0.99, 2 * error_rate), 2),
        ["cx"]
    )

    noisy_simulator = AerSimulator(noise_model=noise_model)

    compiled = transpile(circuit, noisy_simulator)

    return noisy_simulator.run(
        compiled,
        shots=shots
    ).result().get_counts()

Start with an error rate of $0.1\%$, then try $1\%$, $5\%$, and $10\%$.

For each run, look at how clearly you can still identify the four expected peaks.

In [ ]:
error_rates = [0.001, 0.01, 0.05, 0.10]

for rate in error_rates:

    # TODO: run shor_qc using run_with_noise().
    noisy_counts = ...

    print(f"Error rate: {rate:.1%}")
    print(noisy_counts)

    # TODO: plot the histogram.

Now quantify what you saw.

Calculate the fraction of all shots that landed on one of the four ideal peak locations:

$$
000,\qquad010,\qquad100,\qquad110.
$$

Then compare this quantity as the error rate increases.

In [ ]:
expected_peaks = {"000", "010", "100", "110"}

for rate in error_rates:

    noisy_counts = run_with_noise(
        shor_qc,
        error_rate=rate,
        shots=4096
    )

    total_shots = sum(noisy_counts.values())

    # TODO:
    # Count how many shots landed at one of the expected peaks.
    peak_shots = ...

    peak_fraction = peak_shots / total_shots

    print(
        f"Error rate {rate:.1%}: "
        f"{peak_fraction:.1%} of measurements at expected peaks"
    )

**Think about your results:**

What happens to the histogram as the error rate increases?

At what point does the period-$4 pattern become difficult to distinguish?

Try increasing the number of shots. Does taking more measurements undo the effect of noisy gates? Why might collecting more samples help with statistical variation while still being unable to repair errors that occurred during the computation?

---

### Challenge D: put the pieces together

In the main notebook, we manually chose

$$
a=2
$$

before running Shor's algorithm.

A more complete implementation should be able to choose a base, perform some classical checks, run quantum period finding when necessary, interpret the measurement, and return factors.

For this final challenge, build a small automated factoring pipeline for

$$
N=15.
$$

Your program should:

1. randomly choose a candidate $a$ with $1<a<15$,
2. calculate $\gcd(a,15)$,
3. immediately return a factor if that GCD is already non-trivial,
4. otherwise check whether your implementation supports that base,
5. run the appropriate quantum period-finding circuit,
6. use `interpret_results()` to search the measurements for factors,
7. try again if the attempt does not produce a complete factor pair.

There is an important simplification in this challenge: **you only constructed quantum modular-multiplication circuits for a small number of bases.** Your program therefore does not need to implement arbitrary modular arithmetic.

For bases your quantum circuits do not support, simply choose another base.

This mirrors the overall structure of Shor's algorithm while keeping the quantum arithmetic within what we developed in this notebook.

In [ ]:
import random

def full_shors_15(max_attempts=10):
    """
    Attempt to factor 15 using classical preprocessing
    and the quantum period-finding circuits developed
    in this notebook.
    """

    N = 15

    for attempt in range(1, max_attempts + 1):

        # TODO 1:
        # Randomly choose an integer a with 1 < a < N.
        a = ...

        print(f"\nAttempt {attempt}: a={a}")

        # TODO 2:
        # Compute gcd(a, N).
        common_factor = ...

        # TODO 3:
        # If the GCD already gives a non-trivial factor,
        # return the factor pair immediately.


        # TODO 4:
        # Select the appropriate quantum circuit if this
        # notebook supports the chosen value of a.
        #
        # We have a circuit for a=2.
        # If you completed Challenge A, you also have a=7.
        #
        # For unsupported bases, continue to another attempt.

        circuit = ...


        # TODO 5:
        # Run the quantum circuit and collect its counts.
        counts = ...


        # TODO 6:
        # Interpret the measurements using interpret_results().
        factors = ...


        # TODO 7:
        # If two factors were recovered, return them.


    print("No factor pair was recovered.")
    return None


result = full_shors_15()
print("\nFinal result:", result)

If your pipeline successfully returns

$$
(3,5),
$$

you have connected the major pieces of Shor's algorithm into one program:

$$
\boxed{
\text{choose }a
\rightarrow
\text{classical GCD check}
\rightarrow
\text{quantum period finding}
\rightarrow
\text{interpret measurement}
\rightarrow
\text{recover factors}
}
$$

There is still an enormous jump between this educational implementation and a general implementation capable of factoring arbitrary integers. In particular, a general implementation needs reversible modular arithmetic capable of constructing $a^x\bmod N$ for arbitrary supported values of $a$ and $N$.

But that complexity lives inside the same overall algorithmic structure you have already built here.

---

### Challenge E: factor a new number

Everything so far has revolved around

$$
N=15.
$$

For this final challenge, you will take the ideas from the notebook and apply them to a number for which we have not already constructed the period-finding circuit:

$$
\boxed{N=21}
$$

Use

$$
\boxed{a=2}.
$$

This time, most of the circuit will not be provided for you.

However, you will **not** be asked to invent a general reversible modular-multiplication circuit from elementary gates. That is a substantial topic of its own. A helper for constructing those operations is provided below.

Everything else follows ideas you have already used in this notebook.

If you understood the previous sections, you have already encountered every major step you need.

#### Step 1: investigate the classical function

Before building a quantum circuit, investigate

$$
f(x)=2^x\bmod21.
$$

Calculate enough consecutive values to identify where the sequence begins repeating.

From your results, determine the period

$$
r.
$$

Do this **before** constructing the quantum circuit. Knowing the answer classically will allow you to check whether your quantum circuit works.

In [ ]:
N = 21
a = 2

# TODO:
# Print enough values of 2^x mod 21 to identify the period.

for x in range(12):
    value = ...
    print(f"x={x}: {value}")

# From the output above:
expected_period = ...

#### Step 2: choose the register sizes

The work register must be able to represent the values used modulo $21$.

Find the smallest $n_{\text{work}}$ satisfying

$$
2^{n_{\text{work}}}>21.
$$

For the counting register, use

$$
n_{\text{count}}=8.
$$

This gives

$$
2^8=256
$$

possible counting-register outcomes.

There is an important difference from our $N=15$ example. Your period for $N=21$ does **not** divide $256$ exactly. Therefore, you should not expect the inverse QFT to produce perfectly positioned peaks corresponding to exact fractions.

This is precisely the situation in which the continued-fraction interpretation becomes useful.

In [ ]:
N = 21
a = 2

n_count = 8

# TODO:
# Determine the minimum number of work qubits required.
n_work = ...

print("Counting qubits:", n_count)
print("Work qubits:", n_work)
print("Counting states:", 2 ** n_count)
print("Work-register states:", 2 ** n_work)

#### Step 3: determine the controlled multiplications

Recall that if

$$
x=x_0 2^0+x_1 2^1+\cdots,
$$

then

$$
a^x
=
\left(a^{2^0}\right)^{x_0}
\left(a^{2^1}\right)^{x_1}
\left(a^{2^2}\right)^{x_2}
\cdots
$$

Therefore, counting qubit $j$ must control multiplication by

$$
a^{2^j}\bmod N.
$$

For our new problem, calculate

$$
2^{2^j}\bmod21
$$

for every counting qubit.

Look carefully at the resulting sequence. Some multipliers may repeat.

In [ ]:
multipliers = []

for j in range(n_count):

    # TODO:
    # Calculate the modular multiplier controlled by
    # counting qubit j.
    multiplier = ...

    multipliers.append(multiplier)

    print(
        f"Counting qubit q{j}: "
        f"multiply work register by {multiplier} mod {N}"
    )

#### Step 4: construct modular multiplication

For $N=15$, we were lucky: multiplication by $2$ and $4$ modulo $15$ could be represented using simple rotations of the four work bits.

That shortcut does not generally exist.

The helper below constructs a reversible gate implementing

$$
|y\rangle
\longrightarrow
|my\bmod N\rangle
$$

for the valid modular values $0\le y<N$.

States outside that range are left unchanged so that the complete operation remains a valid permutation of the computational basis.

You do not need to modify this function.

Your job is to use it correctly when you construct modular exponentiation.

In [ ]:
from qiskit.circuit.library import UnitaryGate
import numpy as np

def modular_multiplication_gate(multiplier, N, n_work):
    """
    Construct a reversible multiplication-by-multiplier mod N gate.

    Valid values 0 <= y < N are mapped to multiplier*y mod N.
    Computational-basis states >= N are left unchanged.
    """

    dimension = 2 ** n_work
    matrix = np.zeros((dimension, dimension), dtype=complex)

    for y in range(dimension):

        if y < N:
            new_y = (multiplier * y) % N
        else:
            new_y = y

        matrix[new_y, y] = 1

    gate = UnitaryGate(
        matrix,
        label=f"×{multiplier} mod {N}"
    )

    return gate

Before using the helper, think about one requirement.

For multiplication modulo $N$ to act as a reversible permutation, the multiplier must be coprime with $N$:

$$
\gcd(m,N)=1.
$$

Check that the multipliers you found satisfy this condition.

Then test one of your multiplication gates on a known input before trusting it inside the complete Shor circuit.

Choose an input $y$, calculate

$$
my\bmod21
$$

yourself, and check whether the quantum circuit produces the same result.

In [ ]:
# TODO:
# Choose one multiplier from your list and one test input.

m = ...
test_input = ...

expected_output = (m * test_input) % N

print("Multiplier:", m)
print("Input:", test_input)
print("Expected output:", expected_output)

# Construct the gate.
mult_gate = modular_multiplication_gate(m, N, n_work)

# OPTIONAL:
# Build a small test circuit and verify that the gate
# performs the transformation you predicted.

#### Step 5: build modular exponentiation

You now know that counting qubit $j$ should control multiplication by

$$
2^{2^j}\bmod21.
$$

The helper below takes a modular-multiplication gate and turns it into a controlled operation.

Complete the loop so that **every counting qubit controls the correct multiplication**.

Remember the structure from the $N=15$ circuit:

$$
|x\rangle|1\rangle
\longrightarrow
|x\rangle|2^x\bmod N\rangle.
$$

In [ ]:
def apply_modular_exponentiation(circuit, counting, work, a, N):
    """
    Apply the controlled operations required to encode a^x mod N.
    """

    n_work = len(work)

    for j, control_qubit in enumerate(counting):

        # TODO:
        # Determine the multiplier associated with this
        # binary digit of x.
        multiplier = ...

        # Build multiplication by that value modulo N.
        mult_gate = modular_multiplication_gate(
            multiplier,
            N,
            n_work
        )

        # Turn it into a controlled operation.
        controlled_gate = mult_gate.control(1)

        # TODO:
        # Append the controlled gate.
        # It acts on:
        #   1. control_qubit
        #   2. every qubit in the work register

#### Step 6: assemble the period-finding circuit

You now have all the pieces.

Build the complete circuit yourself.

Your circuit should perform the same conceptual sequence as the $N=15$ circuit:

$$
\boxed{
\text{initialize}
\rightarrow
\text{superposition}
\rightarrow
\text{modular exponentiation}
\rightarrow
QFT^{-1}
\rightarrow
\text{measurement}
}
$$

Use the functions already developed in the notebook whenever appropriate.

Only the **counting register** should be measured.

In [ ]:
def shors_circuit_21():
    N = 21
    a = 2

    n_count = 8
    n_work = ...

    circuit = QuantumCircuit(
        n_count + n_work,
        n_count
    )

    counting = list(range(n_count))
    work = list(range(n_count, n_count + n_work))

    # TODO 1:
    # Put the counting register into equal superposition.


    # TODO 2:
    # Initialize the work register to |1>.


    # TODO 3:
    # Apply modular exponentiation using the helper
    # you completed above.


    # TODO 4:
    # Apply the inverse QFT to the counting register.


    # TODO 5:
    # Measure only the counting register.


    return circuit


shor_21 = shors_circuit_21()

shor_21.draw("mpl", fold=40)

#### Step 7: run it

You already know how to compile and simulate a Qiskit circuit.

Run your new circuit using several thousand shots and plot the measurement histogram.

Before interpreting individual measurements, look at the overall shape.

You found the period classically in Step 1. Does the histogram contain a pattern consistent with that period?

Remember that this example will not be as perfectly clean as the $N=15$ histogram. The ideal peak locations do not all correspond to exact integers when the period does not divide $2^{n_{\text{count}}}$.

In [ ]:
# TODO:
# Transpile shor_21 for the simulator.

compiled_21 = ...

# TODO:
# Run the circuit. Try 4096 shots.

result_21 = ...

# TODO:
# Extract the counts.

counts_21 = ...

print("Measurement results:")
print(counts_21)

# TODO:
# Plot the histogram.

#### Step 8: recover the period from the quantum measurements

Now comes the real test.

For eight counting qubits, a measured integer $s$ represents

$$
\frac{s}{256}.
$$

Unlike our $N=15$ example, useful measurements will generally not reduce immediately to an exact fraction whose denominator is the period.

Use the `interpret_results()` function from the main notebook as a starting point, but inspect what it produces carefully.

For one of the largest non-zero peaks:

1. convert its bit string to the integer $s$,
2. calculate $s/256$,
3. use `Fraction(...).limit_denominator(N)` to find a nearby simple fraction,
4. inspect its denominator as a candidate for $r$,
5. verify any candidate using

$$
2^r\bmod21=1.
$$

If one measurement gives only part of the information you need, inspect another major peak.

Do not use the period you calculated in Step 1 as the answer here. That value is your **check**. The goal is to see whether you can recover it from the simulated quantum measurements.

In [ ]:
from fractions import Fraction

# TODO:
# Choose one of the major non-zero measurement outcomes
# produced by YOUR quantum circuit.

measured_bits = ...

s = int(measured_bits, 2)
measured_fraction = s / (2 ** n_count)

approximation = Fraction(
    s,
    2 ** n_count
).limit_denominator(N)

print("Measured bits:", measured_bits)
print("Measured integer:", s)
print("Measured fraction:", measured_fraction)
print("Nearby simple fraction:", approximation)
print("Candidate denominator:", approximation.denominator)

# TODO:
# Check whether the candidate denominator is actually
# a period of 2^x mod 21.

#### Step 9: finish the factorization

Once you have recovered a valid period $r$, finish exactly as you did in the main notebook.

First determine whether the period is suitable for factor extraction.

Then calculate

$$
\gcd\left(2^{r/2}-1,21\right)
$$

and

$$
\gcd\left(2^{r/2}+1,21\right).
$$

If everything worked, these calculations should reveal the two non-trivial factors of $21$.

In [ ]:
# TODO:
# Enter the period recovered from the quantum measurements.

r = ...

# Verify it.
print("Period check:", pow(a, r, N))

# TODO:
# Calculate the two GCDs used by Shor's algorithm.

factor_1 = ...
factor_2 = ...

print("Recovered factors:", factor_1, "and", factor_2)
print(
    "Verification:",
    factor_1,
    "*",
    factor_2,
    "=",
    factor_1 * factor_2
)

#### You just generalized the algorithm

If you completed this challenge successfully, compare what you did with the original $N=15$ example.

You were given a new integer and had to determine:

- the behaviour and period of $a^x\bmod N$,
- the required register sizes,
- the modular multipliers associated with each counting qubit,
- how those multipliers fit into controlled modular exponentiation,
- how to assemble the complete period-finding circuit,
- how to recognize approximate QFT peaks,
- how to recover and verify a candidate period,
- and how to turn that period into factors.

The modular-multiplication helper saved you from having to design arbitrary reversible arithmetic at the gate level. Everything surrounding it, however, is the same period-finding structure developed throughout this notebook.

If you could complete this challenge without copying the $N=15$ circuit and simply changing numbers until something worked, you have more than understood the core structure of the algorithm.

---
## Where to go next

You have now gone from manipulating individual qubits to building a complete quantum algorithm. Along the way, you worked with superposition, measurement, single- and multi-qubit gates, entanglement, interference, the quantum Fourier transform, modular arithmetic, quantum period finding, and the classical post-processing that turns Shor's period into factors.

There is still much more to quantum computing, and there are several directions you can take from here.

### Strengthen the fundamentals

Try modifying circuits you already understand. Change an input state, remove a gate, add another qubit, or change the number of measurement shots. Before running the circuit, **predict what you think will happen** and then use the simulator to test your reasoning.

Being able to predict a circuit before running it is one of the best signs that you are developing a useful intuition for quantum computation.

### Learn more quantum algorithms

Shor's algorithm is only one example of how quantum mechanics can be used computationally. Other important algorithms and techniques to explore include:

- **Grover's algorithm**, which demonstrates quantum search and amplitude amplification,
- **quantum phase estimation**, which generalizes ideas that appeared in our period-finding circuit,
- **variational quantum algorithms**, which combine parameterized quantum circuits with classical optimization,
- and **quantum simulation**, one of the original motivations for building quantum computers.

### Go deeper into the mathematics

If you want to understand why quantum algorithms work at a deeper level, continue developing your knowledge of **linear algebra, complex numbers, probability, and Fourier analysis**.

Concepts such as eigenvalues and eigenvectors, tensor products, unitary matrices, inner products, and changes of basis appear throughout quantum computing. The circuit diagrams eventually become another way of expressing this mathematics.

### Keep learning Qiskit

The [IBM Quantum Learning platform](https://learning.quantum.ibm.com/) contains courses, tutorials, and exercises that continue beyond the material covered here.

As you become more comfortable with Qiskit, you can move beyond ideal simulation and explore transpilation, realistic noise, quantum error correction, and execution on available quantum hardware.

### Most importantly: experiment

Do not feel that you need to understand all of quantum computing before building things.

Take circuits from this notebook and break them. Change the gates. Change the initial states. Add measurements. Increase the register sizes. Try different parameters. Predict what should happen, run the experiment, and investigate whenever the result surprises you.

You now have enough background to read a quantum circuit and ask the most important questions:

**What state am I preparing? What transformation am I applying? Why am I applying it? What information will measurement give me?**

Keep asking those questions, and the more advanced material has somewhere to attach.

---